# Búsqueda conjunta de la configuración de AA


In [1]:
# Importaciones

from itertools import product
from sklearn.model_selection import train_test_split

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


In [2]:
# Rutas

from pathlib import Path
import sys


# Ruta raíz del proyecto
RUTA_PROYECTO = (
    Path.cwd()
    .resolve()
    .parent
)


# Comprobar que sea la raíz correcta
if not (
    RUTA_PROYECTO
    / "src"
).is_dir():
    raise FileNotFoundError(
        "No se encontró la carpeta src en:\n"
        f"{RUTA_PROYECTO}"
    )


# Permitir que Python encuentre los módulos de src
if str(RUTA_PROYECTO) not in sys.path:
    sys.path.insert(
        0,
        str(RUTA_PROYECTO),
    )


from src.loaders import cargar_dataset


print(
    "Raíz del proyecto:",
    RUTA_PROYECTO,
)

print(
    "Módulo cargar_dataset importado correctamente."
)


Raíz del proyecto: C:\Users\adanm\Code\tesis
Módulo cargar_dataset importado correctamente.


## 1. Objetivo y protocolo experimental

La función de este notebook es correr todas las configuraciones para AA utilizando una submuestra de cinco mil estrellas seleccionadas del conjunto de prueba.

## 2. Carga de las particiones

In [3]:
# Grilla de configuraciones

REPRESENTACIONES = (
    "medianas_fase_50",
    "medianas_fase_100",
    "interpolacion_50",
    "interpolacion_100",
)

NORMALIZACIONES = (
    "minmax",
    "centered",
    "zscore",
)

INICIALIZACIONES = (
    "uniform",
    "furthest_sum",
    "pca_convex_hull_furthest_sum",
)

VALORES_K = tuple (range(2, 11))

SEMILLA_MODELO = 0

## Selección reproducible de 5.000 estrellas

La búsqueda conjunta se realiza sobre una muestra estratificada de 5.000
estrellas físicas pertenecientes a la partición de entrenamiento. Cuando
una estrella cuenta con curvas de OGLE-III y OGLE-IV, ambas observaciones
se conservan en la muestra. La selección utiliza la semilla 42 y se
mantiene fija para las 324 configuraciones.

In [4]:
# Configuración de la muestra

N_ESTRELLAS_MUESTRA = 5_000
SEMILLA_MUESTRA = 42

RUTA_RESULTADOS_GRILLA = (
    RUTA_PROYECTO
    / "resultados"
    / "busqueda_grilla_aa_5000"
)

RUTA_RESULTADOS_GRILLA.mkdir(
    parents=True,
    exist_ok=True,
)

RUTA_MANIFIESTO_MUESTRA = (
    RUTA_RESULTADOS_GRILLA
    / "muestra_estrellas_5000.csv"
)


# ----------------------------------------------------------
# Recuperar las estrellas disponibles en entrenamiento
# ----------------------------------------------------------

datos_referencia = cargar_dataset(
    "medianas_fase_50"
)

metadata_train = (
    datos_referencia["train"]["metadata"]
    .copy()
)


# Cada estrella debe tener un solo subtipo
subtipos_por_estrella = (
    metadata_train
    .groupby("star_id_base")["subtipo_rrlyrae"]
    .nunique()
)

if (subtipos_por_estrella > 1).any():
    raise ValueError(
        "Hay estrellas de entrenamiento asociadas "
        "con más de un subtipo."
    )


# Una fila por estrella física
estrellas_train = (
    metadata_train
    .groupby(
        "star_id_base",
        as_index=False,
    )
    .agg(
        subtipo_rrlyrae=(
            "subtipo_rrlyrae",
            "first",
        ),
        n_curvas=(
            "observacion_id",
            "nunique",
        ),
    )
)

if len(estrellas_train) < N_ESTRELLAS_MUESTRA:
    raise ValueError(
        "No existen suficientes estrellas "
        "para construir la muestra."
    )


# ----------------------------------------------------------
# Selección estratificada
# ----------------------------------------------------------

ids_muestra, _ = train_test_split(
    estrellas_train["star_id_base"],
    train_size=N_ESTRELLAS_MUESTRA,
    random_state=SEMILLA_MUESTRA,
    stratify=estrellas_train["subtipo_rrlyrae"],
)

ids_muestra = set(
    ids_muestra
)


manifiesto_muestra = (
    estrellas_train.loc[
        estrellas_train["star_id_base"].isin(
            ids_muestra
        )
    ]
    .sort_values(
        [
            "subtipo_rrlyrae",
            "star_id_base",
        ]
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Comprobaciones
# ----------------------------------------------------------

if manifiesto_muestra["star_id_base"].nunique() != 5_000:
    raise ValueError(
        "La muestra no contiene exactamente "
        "5.000 estrellas físicas."
    )

metadata_curvas_muestra = (
    metadata_train.loc[
        metadata_train["star_id_base"].isin(
            ids_muestra
        )
    ]
    .copy()
)

if not metadata_curvas_muestra["particion"].eq("train").all():
    raise ValueError(
        "La muestra contiene observaciones "
        "fuera de entrenamiento."
    )


# ----------------------------------------------------------
# Guardar el manifiesto
# ----------------------------------------------------------

manifiesto_muestra.to_csv(
    RUTA_MANIFIESTO_MUESTRA,
    index=False,
)


print(
    "Estrellas físicas:",
    f"{manifiesto_muestra['star_id_base'].nunique():,}",
)

print(
    "Curvas incluidas:",
    f"{len(metadata_curvas_muestra):,}",
)

print(
    "Manifiesto guardado en:",
    RUTA_MANIFIESTO_MUESTRA,
)

display(
    manifiesto_muestra[
        "subtipo_rrlyrae"
    ]
    .value_counts()
    .rename("n_estrellas")
    .to_frame()
)

display(
    metadata_curvas_muestra[
        "subtipo_rrlyrae"
    ]
    .value_counts()
    .rename("n_curvas")
    .to_frame()
)

Estrellas físicas: 5,000
Curvas incluidas: 7,295
Manifiesto guardado en: C:\Users\adanm\Code\tesis\resultados\busqueda_grilla_aa_5000\muestra_estrellas_5000.csv


,n_estrellas
subtipo_rrlyrae,
RRab,3535
RRc,1299
RRd,166


,n_curvas
subtipo_rrlyrae,
RRab,5227
RRc,1819
RRd,249


In [5]:
# Funciones para ejecutar la grilla de AA

from time import perf_counter

from src.metricas import (
    ami_metric,
    mse_metric,
)

from src.modelos import (
    aa_model,
    aa_reconstruction,
)

from src.normalizacion import (
    aplicar_normalizacion,
)


# Parámetros fijos de AA
MAX_ITER_AA = 300
TOL_AA = 1e-4

PARAMETROS_NNLS = {
    "max_iter_optimizer": 1_000,
    "const": 100.0,
}


# Archivo donde se guardará una fila por ejecución
RUTA_REGISTRO_GRILLA = (
    RUTA_RESULTADOS_GRILLA
    / "registro_grilla_aa.csv"
)


COLUMNAS_REGISTRO = [
    "configuracion_id",
    "representacion",
    "normalizacion",
    "inicializacion",
    "K",
    "seed_modelo",
    "n_estrellas_train",
    "n_curvas_train",
    "n_curvas_val",
    "n_variables",
    "convergio",
    "iteraciones_reales",
    "mse_train_normalizado",
    "mse_val_normalizado",
    "mse_val_original",
    "ami_val",
    "n_grupos_efectivos_val",
    "n_grupo_menor_val",
    "porcentaje_grupo_menor_val",
    "tiempo_entrenamiento_segundos",
    "tiempo_proyeccion_segundos",
    "tiempo_total_segundos",
    "estado",
    "error",
]


def desnormalizar_reconstruccion(
    X_original,
    X_reconstruida_normalizada,
    metodo,
):
    """Devolver la reconstrucción a la escala original."""

    X_original = np.asarray(
        X_original,
        dtype=np.float64,
    )

    X_reconstruida_normalizada = np.asarray(
        X_reconstruida_normalizada,
        dtype=np.float64,
    )

    if metodo == "minmax":

        minimo = X_original.min(
            axis=1,
            keepdims=True,
        )

        rango = (
            X_original.max(
                axis=1,
                keepdims=True,
            )
            - minimo
        )

        return (
            X_reconstruida_normalizada
            * rango
            + minimo
        )

    if metodo == "centered":

        media = X_original.mean(
            axis=1,
            keepdims=True,
        )

        return (
            X_reconstruida_normalizada
            + media
        )

    if metodo == "zscore":

        media = X_original.mean(
            axis=1,
            keepdims=True,
        )

        desviacion = X_original.std(
            axis=1,
            keepdims=True,
            ddof=0,
        )

        return (
            X_reconstruida_normalizada
            * desviacion
            + media
        )

    raise ValueError(
        f"Normalización no reconocida: {metodo}"
    )


def ejecutar_configuracion_aa(
    representacion,
    normalizacion,
    inicializacion,
    k,
    X_train,
    X_val,
    X_val_original,
    metadata_train_muestra,
    subtipos_val,
):
    """Entrenar y evaluar una configuración de AA."""

    configuracion_id = (
        f"{representacion}"
        f"__{normalizacion}"
        f"__{inicializacion}"
        f"__K{k}"
        f"__seed{SEMILLA_MODELO}"
        f"__muestra5000"
    )

    registro = {
        "configuracion_id": configuracion_id,
        "representacion": representacion,
        "normalizacion": normalizacion,
        "inicializacion": inicializacion,
        "K": k,
        "seed_modelo": SEMILLA_MODELO,
        "n_estrellas_train": (
            metadata_train_muestra[
                "star_id_base"
            ].nunique()
        ),
        "n_curvas_train": X_train.shape[0],
        "n_curvas_val": X_val.shape[0],
        "n_variables": X_train.shape[1],
        "estado": "ERROR",
        "error": None,
    }

    inicio_total = perf_counter()

    try:

        inicio_entrenamiento = perf_counter()

        modelo = aa_model(
            X_train=X_train,
            k=k,
            seed=SEMILLA_MODELO,
            inicializacion=inicializacion,
            n_init=1,
            max_iter=MAX_ITER_AA,
            tol=TOL_AA,
            method="nnls",
            method_params=PARAMETROS_NNLS,
        )

        tiempo_entrenamiento = (
            perf_counter()
            - inicio_entrenamiento
        )

        (
            _,
            _,
            X_train_reconstruida,
        ) = aa_reconstruction(
            modelo=modelo,
            X=X_train,
            coefficients=modelo.coefficients_,
        )

        inicio_proyeccion = perf_counter()

        (
            _,
            grupos_val,
            X_val_reconstruida,
        ) = aa_reconstruction(
            modelo=modelo,
            X=X_val,
        )

        tiempo_proyeccion = (
            perf_counter()
            - inicio_proyeccion
        )

        X_val_reconstruida_original = (
            desnormalizar_reconstruccion(
                X_original=X_val_original,
                X_reconstruida_normalizada=(
                    X_val_reconstruida
                ),
                metodo=normalizacion,
            )
        )

        conteos_grupos = np.bincount(
            grupos_val,
            minlength=k,
        )

        registro.update(
            {
                "convergio": (
                    modelo.n_iter_
                    < MAX_ITER_AA
                ),
                "iteraciones_reales": (
                    modelo.n_iter_
                ),
                "mse_train_normalizado": (
                    mse_metric(
                        X_train,
                        X_train_reconstruida,
                    )
                ),
                "mse_val_normalizado": (
                    mse_metric(
                        X_val,
                        X_val_reconstruida,
                    )
                ),
                "mse_val_original": (
                    mse_metric(
                        X_val_original,
                        X_val_reconstruida_original,
                    )
                ),
                "ami_val": ami_metric(
                    subtipos_val,
                    grupos_val,
                ),
                "n_grupos_efectivos_val": int(
                    (
                        conteos_grupos
                        > 0
                    ).sum()
                ),
                "n_grupo_menor_val": int(
                    conteos_grupos.min()
                ),
                "porcentaje_grupo_menor_val": (
                    conteos_grupos.min()
                    / len(grupos_val)
                    * 100
                ),
                "tiempo_entrenamiento_segundos": (
                    tiempo_entrenamiento
                ),
                "tiempo_proyeccion_segundos": (
                    tiempo_proyeccion
                ),
                "tiempo_total_segundos": (
                    perf_counter()
                    - inicio_total
                ),
                "estado": "OK",
                "error": None,
            }
        )

    except Exception as error:

        registro.update(
            {
                "tiempo_total_segundos": (
                    perf_counter()
                    - inicio_total
                ),
                "estado": "ERROR",
                "error": repr(error),
            }
        )

    return registro


def guardar_registro_grilla(
    registro,
):
    """Agregar una ejecución al archivo de resultados."""

    fila = (
        pd.DataFrame(
            [registro]
        )
        .reindex(
            columns=COLUMNAS_REGISTRO
        )
    )

    fila.to_csv(
        RUTA_REGISTRO_GRILLA,
        mode="a",
        header=(
            not RUTA_REGISTRO_GRILLA.exists()
        ),
        index=False,
    )

Using NumPy backend


## Ejecución de la grilla completa

Se entrenan las 324 configuraciones resultantes de combinar cuatro
representaciones, tres normalizaciones, tres inicializaciones y valores
de \(K\) entre 2 y 10. Todas las ejecuciones utilizan la misma muestra
de 5.000 estrellas y la semilla de modelo 0. Los resultados se guardan
después de cada ejecución para permitir la reanudación del proceso.

In [21]:
# Ejecución de las 324 configuraciones

import gc


# ----------------------------------------------------------
# Cargar la muestra
# ----------------------------------------------------------

manifiesto_muestra = pd.read_csv(
    RUTA_MANIFIESTO_MUESTRA
)

ids_estrellas_muestra = set(
    manifiesto_muestra[
        "star_id_base"
    ]
)


# ----------------------------------------------------------
# Recuperar configuraciones completadas
# ----------------------------------------------------------

if RUTA_REGISTRO_GRILLA.exists():

    registro_existente = pd.read_csv(
        RUTA_REGISTRO_GRILLA
    )

    configuraciones_completadas = set(
        registro_existente.loc[
            registro_existente["estado"].eq("OK"),
            "configuracion_id",
        ]
    )

else:

    configuraciones_completadas = set()


TOTAL_CONFIGURACIONES = (
    len(REPRESENTACIONES)
    * len(NORMALIZACIONES)
    * len(INICIALIZACIONES)
    * len(VALORES_K)
)

print(
    "Configuraciones totales:",
    TOTAL_CONFIGURACIONES,
)

print(
    "Configuraciones ya completadas:",
    len(configuraciones_completadas),
)


# ----------------------------------------------------------
# Recorrer la grilla
# ----------------------------------------------------------

numero_configuracion = 0

for representacion in REPRESENTACIONES:

    datos = cargar_dataset(
        representacion
    )

    metadata_train_completo = (
        datos["train"]["metadata"]
    )

    mascara_muestra = (
        metadata_train_completo[
            "star_id_base"
        ]
        .isin(ids_estrellas_muestra)
        .to_numpy()
    )

    X_train_original = (
        datos["train"]["X"][
            mascara_muestra
        ]
    )

    metadata_train_muestra = (
        metadata_train_completo.loc[
            mascara_muestra
        ]
        .reset_index(drop=True)
    )

    X_val_original = (
        datos["val"]["X"]
    )

    subtipos_val = (
        datos["val"]["metadata"][
            "subtipo_rrlyrae"
        ]
        .to_numpy()
    )

    for normalizacion in NORMALIZACIONES:

        X_train = aplicar_normalizacion(
            X_train_original,
            normalizacion,
        )

        X_val = aplicar_normalizacion(
            X_val_original,
            normalizacion,
        )

        for inicializacion in INICIALIZACIONES:

            for k in VALORES_K:

                numero_configuracion += 1

                configuracion_id = (
                    f"{representacion}"
                    f"__{normalizacion}"
                    f"__{inicializacion}"
                    f"__K{k}"
                    f"__seed{SEMILLA_MODELO}"
                    f"__muestra5000"
                )

                if (
                    configuracion_id
                    in configuraciones_completadas
                ):
                    continue

                print()
                print(
                    f"[{numero_configuracion}/"
                    f"{TOTAL_CONFIGURACIONES}]"
                )

                print(
                    configuracion_id
                )

                registro = ejecutar_configuracion_aa(
                    representacion=representacion,
                    normalizacion=normalizacion,
                    inicializacion=inicializacion,
                    k=k,
                    X_train=X_train,
                    X_val=X_val,
                    X_val_original=X_val_original,
                    metadata_train_muestra=(
                        metadata_train_muestra
                    ),
                    subtipos_val=subtipos_val,
                )

                guardar_registro_grilla(
                    registro
                )

                print(
                    "Estado:",
                    registro["estado"],
                )

                print(
                    "Tiempo total:",
                    round(
                        registro[
                            "tiempo_total_segundos"
                        ],
                        2,
                    ),
                    "segundos",
                )

                if registro["estado"] == "ERROR":

                    print(
                        "Detalle:",
                        registro["error"],
                    )

                    raise RuntimeError(
                        "La grilla se detuvo porque "
                        "una configuración produjo un error."
                    )

                configuraciones_completadas.add(
                    configuracion_id
                )

                gc.collect()

    gc.collect()


# ----------------------------------------------------------
# Resumen final
# ----------------------------------------------------------

df_grilla = pd.read_csv(
    RUTA_REGISTRO_GRILLA
)

print()
print("=" * 60)
print("Ejecución de la grilla finalizada")
print("=" * 60)

print(
    "Filas registradas:",
    len(df_grilla),
)

display(
    df_grilla[
        "estado"
    ]
    .value_counts()
    .rename("n_ejecuciones")
    .to_frame()
)

print(
    "Registro guardado en:",
    RUTA_REGISTRO_GRILLA,
)

Configuraciones totales: 324
Configuraciones ya completadas: 0

[1/324]
medianas_fase_50__minmax__uniform__K2__seed0__muestra5000
Estado: OK
Tiempo total: 6.83 segundos

[2/324]
medianas_fase_50__minmax__uniform__K3__seed0__muestra5000
Estado: OK
Tiempo total: 13.93 segundos

[3/324]
medianas_fase_50__minmax__uniform__K4__seed0__muestra5000
Estado: OK
Tiempo total: 20.29 segundos

[4/324]
medianas_fase_50__minmax__uniform__K5__seed0__muestra5000
Estado: OK
Tiempo total: 14.41 segundos

[5/324]
medianas_fase_50__minmax__uniform__K6__seed0__muestra5000
Estado: OK
Tiempo total: 37.98 segundos

[6/324]
medianas_fase_50__minmax__uniform__K7__seed0__muestra5000
Estado: OK
Tiempo total: 73.99 segundos

[7/324]
medianas_fase_50__minmax__uniform__K8__seed0__muestra5000
Estado: OK
Tiempo total: 107.88 segundos

[8/324]
medianas_fase_50__minmax__uniform__K9__seed0__muestra5000
Estado: OK
Tiempo total: 76.34 segundos

[9/324]
medianas_fase_50__minmax__uniform__K10__seed0__muestra5000
Estado: OK
Ti

,n_ejecuciones
estado,
OK,324


Registro guardado en: C:\Users\adanm\Code\tesis\resultados\busqueda_grilla_aa_5000\registro_grilla_aa.csv


In [19]:
# Resumen de la ejecución completa

df_grilla = pd.read_csv(
    RUTA_REGISTRO_GRILLA
)

# Si alguna configuración fue reintentada,
# conservar el registro más reciente.
df_grilla = (
    df_grilla
    .drop_duplicates(
        subset="configuracion_id",
        keep="last",
    )
    .reset_index(drop=True)
)


print(
    "Configuraciones únicas:",
    len(df_grilla),
)

print(
    "\nEstados:"
)

display(
    df_grilla[
        "estado"
    ]
    .value_counts(
        dropna=False
    )
    .rename("n_ejecuciones")
    .to_frame()
)


ejecuciones_ok = (
    df_grilla.loc[
        df_grilla["estado"] == "OK"
    ]
    .copy()
)


print(
    "Ejecuciones correctas:",
    len(ejecuciones_ok),
)

print(
    "Ejecuciones que convergieron:",
    int(
        ejecuciones_ok[
            "convergio"
        ].sum()
    ),
)

print(
    "Ejecuciones que alcanzaron "
    "el máximo de iteraciones:",
    int(
        (~ejecuciones_ok["convergio"])
        .sum()
    ),
)

print(
    "Ejecuciones con menos grupos "
    "efectivos que K:",
    int(
        (
            ejecuciones_ok[
                "n_grupos_efectivos_val"
            ]
            < ejecuciones_ok["K"]
        ).sum()
    ),
)


# Mostrar problemas solamente si existen
problemas = df_grilla.loc[
    (df_grilla["estado"] != "OK")
    | (
        df_grilla["convergio"]
        == False
    )
    | (
        df_grilla[
            "n_grupos_efectivos_val"
        ]
        < df_grilla["K"]
    )
]

if len(problemas) > 0:

    display(
        problemas[
            [
                "representacion",
                "normalizacion",
                "inicializacion",
                "K",
                "convergio",
                "iteraciones_reales",
                "n_grupos_efectivos_val",
                "estado",
                "error",
            ]
        ]
    )

else:

    print(
        "\nNo se detectaron problemas "
        "en las 324 configuraciones."
    )

Configuraciones únicas: 324

Estados:


,n_ejecuciones
estado,
OK,324


Ejecuciones correctas: 324
Ejecuciones que convergieron: 246
Ejecuciones que alcanzaron el máximo de iteraciones: 78
Ejecuciones con menos grupos efectivos que K: 34


,representacion,normalizacion,inicializacion,K,convergio,iteraciones_reales,n_grupos_efectivos_val,estado,error
6,medianas_fase_50,minmax,uniform,8,False,300,8,OK,NaN
17,medianas_fase_50,minmax,furthest_sum,10,False,300,10,OK,NaN
30,medianas_fase_50,centered,uniform,5,False,300,5,OK,NaN
32,medianas_fase_50,centered,uniform,7,False,300,7,OK,NaN
39,medianas_fase_50,centered,furthest_sum,5,False,300,5,OK,NaN
...,...,...,...,...,...,...,...,...,...
314,interpolacion_100,zscore,furthest_sum,10,False,300,10,OK,NaN
317,interpolacion_100,zscore,pca_convex_hull_furthest_sum,4,False,300,4,OK,NaN
319,interpolacion_100,zscore,pca_convex_hull_furthest_sum,6,False,300,6,OK,NaN
322,interpolacion_100,zscore,pca_convex_hull_furthest_sum,9,False,300,9,OK,NaN


In [20]:
# ==========================================================
# Comparación de las inicializaciones
# ==========================================================

COLUMNAS_BLOQUE = [
    "representacion",
    "normalizacion",
    "K",
]


# ----------------------------------------------------------
# Resumen general de convergencia
# ----------------------------------------------------------

resumen_convergencia_inicializacion = (
    ejecuciones_ok
    .groupby(
        "inicializacion",
        as_index=False,
    )
    .agg(
        n_ejecuciones=(
            "configuracion_id",
            "count",
        ),
        n_convergieron=(
            "convergio",
            "sum",
        ),
        tiempo_mediano_segundos=(
            "tiempo_entrenamiento_segundos",
            "median",
        ),
    )
)

resumen_convergencia_inicializacion[
    "tasa_convergencia_pct"
] = (
    100
    * resumen_convergencia_inicializacion[
        "n_convergieron"
    ]
    / resumen_convergencia_inicializacion[
        "n_ejecuciones"
    ]
)

resumen_convergencia_inicializacion[
    "n_max_iter"
] = (
    resumen_convergencia_inicializacion[
        "n_ejecuciones"
    ]
    - resumen_convergencia_inicializacion[
        "n_convergieron"
    ]
)


# ----------------------------------------------------------
# Conservar bloques donde convergieron las tres opciones
# ----------------------------------------------------------

ejecuciones_convergentes = (
    ejecuciones_ok.loc[
        ejecuciones_ok["convergio"]
    ]
    .copy()
)

bloques_completos = (
    ejecuciones_convergentes
    .groupby(
        COLUMNAS_BLOQUE
    )["inicializacion"]
    .nunique()
)

bloques_completos = (
    bloques_completos.loc[
        bloques_completos
        == len(INICIALIZACIONES)
    ]
    .index
)

comparacion_inicializaciones = (
    ejecuciones_convergentes
    .set_index(
        COLUMNAS_BLOQUE
    )
    .loc[
        bloques_completos
    ]
    .reset_index()
)


# ----------------------------------------------------------
# Mejor MSE y mejor tiempo dentro de cada bloque
# ----------------------------------------------------------

comparacion_inicializaciones[
    "mejor_mse_bloque"
] = (
    comparacion_inicializaciones
    .groupby(
        COLUMNAS_BLOQUE
    )["mse_val_original"]
    .transform("min")
)

comparacion_inicializaciones[
    "mejor_tiempo_bloque"
] = (
    comparacion_inicializaciones
    .groupby(
        COLUMNAS_BLOQUE
    )["tiempo_entrenamiento_segundos"]
    .transform("min")
)


# Diferencia porcentual respecto del mejor resultado del bloque

comparacion_inicializaciones[
    "diferencia_mse_pct"
] = (
    100
    * (
        comparacion_inicializaciones[
            "mse_val_original"
        ]
        / comparacion_inicializaciones[
            "mejor_mse_bloque"
        ]
        - 1
    )
)

comparacion_inicializaciones[
    "diferencia_tiempo_pct"
] = (
    100
    * (
        comparacion_inicializaciones[
            "tiempo_entrenamiento_segundos"
        ]
        / comparacion_inicializaciones[
            "mejor_tiempo_bloque"
        ]
        - 1
    )
)


# Identificar ganadores admitiendo diferencias numéricas mínimas

comparacion_inicializaciones[
    "mejor_mse"
] = np.isclose(
    comparacion_inicializaciones[
        "mse_val_original"
    ],
    comparacion_inicializaciones[
        "mejor_mse_bloque"
    ],
    rtol=1e-6,
    atol=1e-12,
)

comparacion_inicializaciones[
    "mejor_tiempo"
] = np.isclose(
    comparacion_inicializaciones[
        "tiempo_entrenamiento_segundos"
    ],
    comparacion_inicializaciones[
        "mejor_tiempo_bloque"
    ],
    rtol=1e-6,
    atol=1e-9,
)


# ----------------------------------------------------------
# Resumen de la comparación pareada
# ----------------------------------------------------------

resumen_comparacion_inicializaciones = (
    comparacion_inicializaciones
    .groupby(
        "inicializacion",
        as_index=False,
    )
    .agg(
        n_bloques=(
            "configuracion_id",
            "count",
        ),
        veces_mejor_mse=(
            "mejor_mse",
            "sum",
        ),
        diferencia_mse_mediana_pct=(
            "diferencia_mse_pct",
            "median",
        ),
        veces_menor_tiempo=(
            "mejor_tiempo",
            "sum",
        ),
        diferencia_tiempo_mediana_pct=(
            "diferencia_tiempo_pct",
            "median",
        ),
        tiempo_mediano_segundos=(
            "tiempo_entrenamiento_segundos",
            "median",
        ),
    )
)


# ----------------------------------------------------------
# Mostrar resultados
# ----------------------------------------------------------

print(
    "Bloques totales:",
    4 * 3 * 9,
)

print(
    "Bloques donde convergieron "
    "las tres inicializaciones:",
    len(bloques_completos),
)

print(
    "\nConvergencia por inicialización:"
)

display(
    resumen_convergencia_inicializacion
    .sort_values(
        "tasa_convergencia_pct",
        ascending=False,
    )
    .round(4)
)

print(
    "\nComparación pareada mediante MSE y tiempo:"
)

display(
    resumen_comparacion_inicializaciones
    .sort_values(
        [
            "diferencia_mse_mediana_pct",
            "tiempo_mediano_segundos",
        ]
    )
    .round(4)
)

Bloques totales: 108
Bloques donde convergieron las tres inicializaciones: 67

Convergencia por inicialización:


,inicializacion,n_ejecuciones,n_convergieron,tiempo_mediano_segundos,tasa_convergencia_pct,n_max_iter
0,furthest_sum,108,83,66.2143,76.8519,25
1,pca_convex_hull_furthest_sum,108,82,62.7206,75.9259,26
2,uniform,108,81,69.1975,75.0000,27



Comparación pareada mediante MSE y tiempo:


,inicializacion,n_bloques,veces_mejor_mse,diferencia_mse_mediana_pct,veces_menor_tiempo,diferencia_tiempo_mediana_pct,tiempo_mediano_segundos
2,uniform,67,56,0.0000,10,29.0698,39.3838
0,furthest_sum,67,25,0.0007,31,2.2726,38.3869
1,pca_convex_hull_furthest_sum,67,29,0.0008,26,6.8104,35.7731


In [21]:
# ==========================================================
# Configuraciones válidas y resumen por K
# ==========================================================

configuraciones_validas = (
    ejecuciones_ok.loc[
        ejecuciones_ok["convergio"]
        & (
            ejecuciones_ok["n_grupos_efectivos_val"]
            == ejecuciones_ok["K"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Configuraciones ejecutadas:",
    len(ejecuciones_ok),
)

print(
    "Configuraciones convergentes "
    "con K grupos efectivos:",
    len(configuraciones_validas),
)


# ----------------------------------------------------------
# Resumen general por K
# ----------------------------------------------------------

resumen_por_k = (
    configuraciones_validas
    .groupby(
        "K",
        as_index=False,
    )
    .agg(
        n_configuraciones=(
            "configuracion_id",
            "count",
        ),
        mse_original_minimo=(
            "mse_val_original",
            "min",
        ),
        mse_original_mediano=(
            "mse_val_original",
            "median",
        ),
        ami_maximo=(
            "ami_val",
            "max",
        ),
        ami_mediano=(
            "ami_val",
            "median",
        ),
        tiempo_mediano_segundos=(
            "tiempo_entrenamiento_segundos",
            "median",
        ),
    )
    .sort_values("K")
    .reset_index(drop=True)
)


# Reducción del mejor MSE respecto del K anterior

resumen_por_k[
    "reduccion_mse_minimo_pct"
] = (
    resumen_por_k[
        "mse_original_minimo"
    ]
    .pct_change()
    .mul(-100)
)


# ----------------------------------------------------------
# Mejor configuración por MSE para cada K
# ----------------------------------------------------------

indices_mejor_mse = (
    configuraciones_validas
    .groupby("K")[
        "mse_val_original"
    ]
    .idxmin()
)

mejor_mse_por_k = (
    configuraciones_validas.loc[
        indices_mejor_mse,
        [
            "K",
            "representacion",
            "normalizacion",
            "inicializacion",
            "mse_val_original",
            "ami_val",
            "tiempo_entrenamiento_segundos",
            "iteraciones_reales",
        ],
    ]
    .sort_values("K")
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Mejor configuración por AMI para cada K
# ----------------------------------------------------------

indices_mejor_ami = (
    configuraciones_validas
    .groupby("K")[
        "ami_val"
    ]
    .idxmax()
)

mejor_ami_por_k = (
    configuraciones_validas.loc[
        indices_mejor_ami,
        [
            "K",
            "representacion",
            "normalizacion",
            "inicializacion",
            "ami_val",
            "mse_val_original",
            "tiempo_entrenamiento_segundos",
            "iteraciones_reales",
        ],
    ]
    .sort_values("K")
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Mostrar resultados
# ----------------------------------------------------------

print(
    "\nResumen por número de arquetipos:"
)

display(
    resumen_por_k.round(6)
)


print(
    "\nMejor reconstrucción para cada K:"
)

display(
    mejor_mse_por_k.round(6)
)


print(
    "\nMayor asociación con los subtipos para cada K:"
)

display(
    mejor_ami_por_k.round(6)
)


Configuraciones ejecutadas: 324
Configuraciones convergentes con K grupos efectivos: 222

Resumen por número de arquetipos:


,K,n_configuraciones,mse_original_minimo,mse_original_mediano,ami_maximo,ami_mediano,tiempo_mediano_segundos,reduccion_mse_minimo_pct
0,2,36,0.001195,0.002939,0.616066,0.580538,12.447068,NaN
1,3,36,0.000736,0.002228,0.455755,0.438993,24.657700,38.424196
2,4,30,0.000716,0.002559,0.416138,0.385296,35.458711,2.703750
3,5,22,0.000582,0.002593,0.427747,0.392101,41.862231,18.765233
4,6,27,0.000555,0.001225,0.411420,0.371941,58.629549,4.514056
5,7,24,0.000536,0.001777,0.418808,0.359908,76.741841,3.397289
6,8,19,0.000526,0.002373,0.387549,0.347658,90.868996,1.966793
7,9,17,0.000504,0.001102,0.406584,0.350366,75.857202,4.234188
8,10,11,0.000493,0.000991,0.386076,0.345610,59.993080,2.080382



Mejor reconstrucción para cada K:


,K,representacion,normalizacion,inicializacion,mse_val_original,ami_val,tiempo_entrenamiento_segundos,iteraciones_reales
0,2,medianas_fase_50,zscore,furthest_sum,0.001195,0.612922,5.711839,25
1,3,medianas_fase_50,zscore,uniform,0.000736,0.454134,9.205210,41
2,4,medianas_fase_50,centered,furthest_sum,0.000716,0.322852,15.963257,91
3,5,medianas_fase_50,zscore,uniform,0.000582,0.402825,86.568693,258
4,6,medianas_fase_50,zscore,uniform,0.000555,0.383277,78.004603,200
5,7,medianas_fase_50,centered,furthest_sum,0.000536,0.330621,37.633174,118
6,8,medianas_fase_50,centered,uniform,0.000526,0.333567,39.180315,108
7,9,medianas_fase_50,centered,uniform,0.000504,0.320758,108.265429,252
8,10,medianas_fase_50,centered,uniform,0.000493,0.316378,55.806908,123



Mayor asociación con los subtipos para cada K:


,K,representacion,normalizacion,inicializacion,ami_val,mse_val_original,tiempo_entrenamiento_segundos,iteraciones_reales
0,2,medianas_fase_100,zscore,uniform,0.616066,0.001689,17.374041,29
1,3,medianas_fase_50,minmax,furthest_sum,0.455755,0.001079,12.142620,51
2,4,medianas_fase_50,minmax,uniform,0.416138,0.000930,20.010429,82
3,5,medianas_fase_50,minmax,pca_convex_hull_furthest_sum,0.427747,0.000907,12.597029,46
4,6,medianas_fase_50,minmax,furthest_sum,0.411420,0.000730,66.409902,244
5,7,medianas_fase_50,minmax,pca_convex_hull_furthest_sum,0.418808,0.000701,53.506763,158
6,8,medianas_fase_50,minmax,pca_convex_hull_furthest_sum,0.387549,0.000615,54.201065,146
7,9,medianas_fase_50,zscore,pca_convex_hull_furthest_sum,0.406584,0.000584,94.022146,252
8,10,medianas_fase_50,minmax,pca_convex_hull_furthest_sum,0.386076,0.000594,66.926359,163


In [22]:
# ==========================================================
# Frente de Pareto entre MSE y AMI
# ==========================================================

def obtener_frente_pareto(df):
    """
    Conserva las configuraciones que no son superadas
    simultáneamente en MSE y AMI.
    """

    datos = (
        df
        .copy()
        .reset_index(drop=True)
    )

    mse = datos[
        "mse_val_original"
    ].to_numpy()

    ami = datos[
        "ami_val"
    ].to_numpy()

    es_pareto = np.ones(
        len(datos),
        dtype=bool,
    )

    for indice in range(len(datos)):

        existe_dominante = (
            (mse <= mse[indice])
            & (ami >= ami[indice])
            & (
                (mse < mse[indice])
                | (ami > ami[indice])
            )
        ).any()

        if existe_dominante:
            es_pareto[indice] = False

    return (
        datos.loc[es_pareto]
        .sort_values(
            [
                "K",
                "mse_val_original",
                "ami_val",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )


# ----------------------------------------------------------
# Frente global
# ----------------------------------------------------------

frente_pareto_global = obtener_frente_pareto(
    configuraciones_validas
)


# ----------------------------------------------------------
# Frente correspondiente exclusivamente a K = 5
# ----------------------------------------------------------

configuraciones_k5 = (
    configuraciones_validas.loc[
        configuraciones_validas["K"] == 5
    ]
    .copy()
)

frente_pareto_k5 = obtener_frente_pareto(
    configuraciones_k5
)


columnas_resultado = [
    "representacion",
    "normalizacion",
    "inicializacion",
    "K",
    "mse_val_original",
    "ami_val",
    "tiempo_entrenamiento_segundos",
    "iteraciones_reales",
]


# ----------------------------------------------------------
# Mostrar resultados
# ----------------------------------------------------------

print(
    "Configuraciones del frente global:",
    len(frente_pareto_global),
)

display(
    frente_pareto_global[
        columnas_resultado
    ].round(6)
)


print(
    "\nCantidad de configuraciones del frente "
    "global según K:"
)

display(
    frente_pareto_global[
        "K"
    ]
    .value_counts()
    .sort_index()
    .rename("n_configuraciones")
    .to_frame()
)


print(
    "\nConfiguraciones no dominadas para K = 5:",
    len(frente_pareto_k5),
)

display(
    frente_pareto_k5[
        columnas_resultado
    ].round(6)
)

Configuraciones del frente global: 17


,representacion,normalizacion,inicializacion,K,mse_val_original,ami_val,tiempo_entrenamiento_segundos,iteraciones_reales
0,medianas_fase_50,zscore,furthest_sum,2,0.001195,0.612922,5.711839,25
1,medianas_fase_100,zscore,furthest_sum,2,0.001689,0.616066,12.472215,25
2,medianas_fase_50,zscore,uniform,3,0.000736,0.454134,9.205210,41
3,medianas_fase_50,minmax,uniform,3,0.001079,0.455755,13.505060,54
4,medianas_fase_50,minmax,pca_convex_hull_furthest_sum,3,0.001079,0.455755,13.162795,54
5,medianas_fase_50,zscore,uniform,5,0.000582,0.402825,86.568693,258
6,medianas_fase_50,zscore,pca_convex_hull_furthest_sum,5,0.000582,0.402837,37.857875,117
7,medianas_fase_50,zscore,uniform,6,0.000555,0.383277,78.004603,200
8,medianas_fase_50,zscore,furthest_sum,7,0.000568,0.397819,93.733205,273
9,medianas_fase_50,minmax,pca_convex_hull_furthest_sum,7,0.000701,0.418808,53.506763,158



Cantidad de configuraciones del frente global según K:


,n_configuraciones
K,
2,2
3,3
5,2
6,1
7,2
9,4
10,3



Configuraciones no dominadas para K = 5: 4


,representacion,normalizacion,inicializacion,K,mse_val_original,ami_val,tiempo_entrenamiento_segundos,iteraciones_reales
0,medianas_fase_50,zscore,uniform,5,0.000582,0.402825,86.568693,258
1,medianas_fase_50,zscore,pca_convex_hull_furthest_sum,5,0.000582,0.402837,37.857875,117
2,medianas_fase_50,minmax,uniform,5,0.000781,0.402984,14.047262,54
3,medianas_fase_50,minmax,pca_convex_hull_furthest_sum,5,0.000907,0.427747,12.597029,46


In [26]:
# ==========================================================
# Guardar resultados y configuración seleccionada
# ==========================================================

import json


# ----------------------------------------------------------
# Directorios de salida
# ----------------------------------------------------------

RUTA_TABLAS_GRILLA = (
    RUTA_RESULTADOS_GRILLA
    / "tablas"
)

RUTA_TABLAS_GRILLA.mkdir(
    parents=True,
    exist_ok=True,
)


# ----------------------------------------------------------
# Identificar inequívocamente la configuración seleccionada
# ----------------------------------------------------------

configuracion_seleccionada = (
    configuraciones_validas.loc[
        (
            configuraciones_validas["representacion"]
            == "medianas_fase_50"
        )
        & (
            configuraciones_validas["normalizacion"]
            == "zscore"
        )
        & (
            configuraciones_validas["inicializacion"]
            == "pca_convex_hull_furthest_sum"
        )
        & (
            configuraciones_validas["K"]
            == 5
        )
    ]
    .copy()
    .reset_index(drop=True)
)

if len(configuracion_seleccionada) != 1:
    raise ValueError(
        "No se encontró exactamente una configuración seleccionada."
    )


# ----------------------------------------------------------
# Preparar comparación de candidatos para K = 5
# ----------------------------------------------------------

comparacion_candidatos_k5 = (
    frente_pareto_k5
    .copy()
    .reset_index(drop=True)
)

comparacion_candidatos_k5[
    "rmse_val_original"
] = np.sqrt(
    comparacion_candidatos_k5[
        "mse_val_original"
    ]
)

mse_seleccionado = float(
    configuracion_seleccionada.loc[
        0,
        "mse_val_original",
    ]
)

ami_seleccionado = float(
    configuracion_seleccionada.loc[
        0,
        "ami_val",
    ]
)

tiempo_seleccionado = float(
    configuracion_seleccionada.loc[
        0,
        "tiempo_entrenamiento_segundos",
    ]
)

comparacion_candidatos_k5[
    "diferencia_mse_respecto_seleccion_pct"
] = (
    100
    * (
        comparacion_candidatos_k5[
            "mse_val_original"
        ]
        / mse_seleccionado
        - 1
    )
)

comparacion_candidatos_k5[
    "diferencia_ami_respecto_seleccion"
] = (
    comparacion_candidatos_k5[
        "ami_val"
    ]
    - ami_seleccionado
)

comparacion_candidatos_k5[
    "diferencia_tiempo_respecto_seleccion_pct"
] = (
    100
    * (
        comparacion_candidatos_k5[
            "tiempo_entrenamiento_segundos"
        ]
        / tiempo_seleccionado
        - 1
    )
)


# ----------------------------------------------------------
# Guardar tablas
# ----------------------------------------------------------

tablas_guardar = {
    "configuraciones_validas.csv": (
        configuraciones_validas
    ),
    "resumen_convergencia_inicializacion.csv": (
        resumen_convergencia_inicializacion
    ),
    "comparacion_inicializaciones.csv": (
        resumen_comparacion_inicializaciones
    ),
    "resumen_por_k.csv": (
        resumen_por_k
    ),
    "mejor_mse_por_k.csv": (
        mejor_mse_por_k
    ),
    "mejor_ami_por_k.csv": (
        mejor_ami_por_k
    ),
    "frente_pareto_global.csv": (
        frente_pareto_global
    ),
    "frente_pareto_k5.csv": (
        frente_pareto_k5
    ),
    "comparacion_candidatos_k5.csv": (
        comparacion_candidatos_k5
    ),
    "seleccion_configuracion_aa.csv": (
        configuracion_seleccionada
    ),
}

for nombre_archivo, tabla in tablas_guardar.items():

    tabla.to_csv(
        RUTA_TABLAS_GRILLA
        / nombre_archivo,
        index=False,
    )


# ----------------------------------------------------------
# Guardar descripción reproducible de la búsqueda
# ----------------------------------------------------------

fila_seleccionada = (
    configuracion_seleccionada
    .iloc[0]
)

resumen_busqueda = {
    "muestra": {
        "n_estrellas": int(
            manifiesto_muestra[
                "star_id_base"
            ].nunique()
        ),
        "n_curvas": int(
            len(
                metadata_curvas_muestra
            )
        ),
        "semilla_muestra": int(
            SEMILLA_MUESTRA
        ),
        "particion": "train",
    },
    "grilla": {
        "representaciones": list(
            REPRESENTACIONES
        ),
        "normalizaciones": list(
            NORMALIZACIONES
        ),
        "inicializaciones": list(
            INICIALIZACIONES
        ),
        "valores_k": list(
            VALORES_K
        ),
        "semilla_modelo": int(
            SEMILLA_MODELO
        ),
        "n_configuraciones": int(
            len(ejecuciones_ok)
        ),
        "n_configuraciones_validas": int(
            len(configuraciones_validas)
        ),
    },
    "optimizacion": {
        "metodo": "nnls",
        "max_iter_aa": 300,
        "tolerancia_aa": 1e-4,
        "max_iter_nnls": 1000,
        "constante_nnls": 100.0,
    },
    "configuracion_seleccionada": {
        "representacion": str(
            fila_seleccionada[
                "representacion"
            ]
        ),
        "normalizacion": str(
            fila_seleccionada[
                "normalizacion"
            ]
        ),
        "inicializacion": str(
            fila_seleccionada[
                "inicializacion"
            ]
        ),
        "K": int(
            fila_seleccionada["K"]
        ),
        "mse_val_original": float(
            fila_seleccionada[
                "mse_val_original"
            ]
        ),
        "ami_val": float(
            fila_seleccionada[
                "ami_val"
            ]
        ),
        "tiempo_entrenamiento_segundos": float(
            fila_seleccionada[
                "tiempo_entrenamiento_segundos"
            ]
        ),
        "iteraciones_reales": int(
            fila_seleccionada[
                "iteraciones_reales"
            ]
        ),
    },
    "criterio_seleccion": [
        (
            "Medianas de fase con 50 valores presentó "
            "los mejores resultados globales de reconstrucción."
        ),
        (
            "K=5 fue el último valor con una reducción "
            "sustancial del MSE."
        ),
        (
            "Z-score obtuvo el menor MSE original entre "
            "las configuraciones candidatas con K=5."
        ),
        (
            "La inicialización PCA, envolvente convexa "
            "y Furthest Sum alcanzó prácticamente el mismo "
            "MSE que Uniform, con menos tiempo e iteraciones."
        ),
    ],
}

RUTA_RESUMEN_BUSQUEDA = (
    RUTA_RESULTADOS_GRILLA
    / "configuracion_busqueda_grilla.json"
)

with RUTA_RESUMEN_BUSQUEDA.open(
    "w",
    encoding="utf-8",
) as archivo_json:

    json.dump(
        resumen_busqueda,
        archivo_json,
        ensure_ascii=False,
        indent=4,
    )


# ----------------------------------------------------------
# Comprobar resultados guardados
# ----------------------------------------------------------

print(
    "Directorio de resultados:",
    RUTA_RESULTADOS_GRILLA,
)

print(
    "Tablas guardadas:",
    len(tablas_guardar),
)

print(
    "Configuración documentada en:",
    RUTA_RESUMEN_BUSQUEDA,
)

print(
    "\nConfiguración seleccionada:"
)

display(
    configuracion_seleccionada[
        [
            "representacion",
            "normalizacion",
            "inicializacion",
            "K",
            "mse_val_original",
            "ami_val",
            "tiempo_entrenamiento_segundos",
            "iteraciones_reales",
        ]
    ].round(9)
)

Directorio de resultados: C:\Users\adanm\Code\tesis\resultados\busqueda_grilla_aa_5000
Tablas guardadas: 10
Configuración documentada en: C:\Users\adanm\Code\tesis\resultados\busqueda_grilla_aa_5000\configuracion_busqueda_grilla.json

Configuración seleccionada:


,representacion,normalizacion,inicializacion,K,mse_val_original,ami_val,tiempo_entrenamiento_segundos,iteraciones_reales
0,medianas_fase_50,zscore,pca_convex_hull_furthest_sum,5,0.000582,0.402837,37.857875,117


In [3]:
# ==========================================================
# Ejecutar la configuración final sobre el dataset completo
# ==========================================================

# Permitir las importaciones internas utilizadas por src

RUTA_SRC = (
    RUTA_PROYECTO
    / "src"
)

if str(RUTA_SRC) not in sys.path:
    sys.path.insert(
        0,
        str(RUTA_SRC),
    )


from experimento_aa import run_aa_experiment


# ----------------------------------------------------------
# Configuración seleccionada
# ----------------------------------------------------------

DATASET_FINAL = "medianas_fase_50"
NORMALIZACION_FINAL = "zscore"

INICIALIZACION_FINAL = (
    "pca_convex_hull_furthest_sum"
)

K_FINAL = 5

SEMILLAS_FINALES = (
    0,
    1,
    2,
    3,
    4,
)


# ----------------------------------------------------------
# Ejecutar las cinco semillas
# ----------------------------------------------------------

resultados_familia_final = []

for semilla in SEMILLAS_FINALES:

    print(
        "\n"
        + "=" * 60
    )

    print(
        "Ejecutando semilla:",
        semilla,
    )

    print(
        "=" * 60
    )

    resultado = run_aa_experiment(
        dataset=DATASET_FINAL,
        normalizacion=NORMALIZACION_FINAL,
        k=K_FINAL,
        seed=semilla,
        inicializacion=INICIALIZACION_FINAL,
    )

    resultados_familia_final.append(
        resultado
    )


# ----------------------------------------------------------
# Recuperar las ejecuciones desde el registro general
# ----------------------------------------------------------

RUTA_REGISTRO_GENERAL = (
    RUTA_PROYECTO
    / "resultados"
    / "registro_experimentos.csv"
)

df_registro_actualizado = pd.read_csv(
    RUTA_REGISTRO_GENERAL
)

familia_aa_final = (
    df_registro_actualizado.loc[
        (
            df_registro_actualizado["modelo"]
            == "archetypal_analysis"
        )
        & (
            df_registro_actualizado["dataset"]
            == DATASET_FINAL
        )
        & (
            df_registro_actualizado["normalizacion"]
            == NORMALIZACION_FINAL
        )
        & (
            df_registro_actualizado["inicializacion"]
            == INICIALIZACION_FINAL
        )
        & (
            df_registro_actualizado["K"]
            == K_FINAL
        )
        & (
            df_registro_actualizado[
                "seed_modelo"
            ].isin(
                SEMILLAS_FINALES
            )
        )
    ]
    .drop_duplicates(
        subset="experiment_id",
        keep="last",
    )
    .sort_values(
        "seed_modelo"
    )
    .reset_index(drop=True)
)


if len(familia_aa_final) != 5:
    raise ValueError(
        "No se encontraron exactamente cinco "
        "ejecuciones de la configuración final."
    )


print(
    "\nEjecuciones recuperadas:",
    len(familia_aa_final),
)

display(
    familia_aa_final[
        [
            "experiment_id",
            "seed_modelo",
            "convergio",
            "iteraciones_reales",
            "tiempo_entrenamiento_segundos",
            "mse_val",
            "purity_val",
            "ami_val",
        ]
    ]
)

Using NumPy backend

Ejecutando semilla: 0
Experimento completado: 1b36fa390c8a9c63
Tiempo de entrenamiento: 656.261 segundos
Tiempo total: 659.242 segundos
RSS validación: 42984.547860

Ejecutando semilla: 1
Experimento completado: f65d733d751dea5f
Tiempo de entrenamiento: 1414.647 segundos
Tiempo total: 1416.546 segundos
RSS validación: 40287.064848

Ejecutando semilla: 2
Experimento completado: c684c65cfce986f0
Tiempo de entrenamiento: 597.595 segundos
Tiempo total: 599.047 segundos
RSS validación: 42984.547860

Ejecutando semilla: 3
Experimento completado: b55cf869ec2d9132
Tiempo de entrenamiento: 534.351 segundos
Tiempo total: 535.514 segundos
RSS validación: 42984.547860

Ejecutando semilla: 4
Experimento completado: 1434f4b06059bbdd
Tiempo de entrenamiento: 1336.041 segundos
Tiempo total: 1337.475 segundos
RSS validación: 40287.064554

Ejecuciones recuperadas: 5


,experiment_id,seed_modelo,convergio,iteraciones_reales,tiempo_entrenamiento_segundos,mse_val,purity_val,ami_val
0,1b36fa390c8a9c63,0,True,177,656.260943,0.041781,0.917671,0.451175
1,f65d733d751dea5f,1,False,300,1414.646682,0.039159,0.925447,0.404173
2,c684c65cfce986f0,2,True,177,597.594655,0.041781,0.917671,0.451175
3,b55cf869ec2d9132,3,True,177,534.350739,0.041781,0.917671,0.451175
4,1434f4b06059bbdd,4,False,300,1336.041368,0.039159,0.925447,0.404173


In [4]:
# ==========================================================
# Estabilidad de la configuración final entre semillas
# ==========================================================

from itertools import combinations

from sklearn.metrics import adjusted_rand_score


# ----------------------------------------------------------
# Cargar los grupos de validación
# ----------------------------------------------------------

grupos_validacion_por_semilla = {}

for _, fila in familia_aa_final.iterrows():

    semilla = int(
        fila["seed_modelo"]
    )

    ruta_experimento = (
        RUTA_PROYECTO
        / Path(
            fila["ruta_experimento"]
        )
    )

    ruta_grupos = (
        ruta_experimento
        / "grupos_val.npy"
    )

    if not ruta_grupos.exists():
        raise FileNotFoundError(
            "No se encontraron los grupos de validación en:\n"
            f"{ruta_grupos}"
        )

    grupos_validacion_por_semilla[
        semilla
    ] = np.load(
        ruta_grupos
    )


# ----------------------------------------------------------
# Comparaciones pareadas mediante ARI
# ----------------------------------------------------------

comparaciones_ari = []

for semilla_1, semilla_2 in combinations(
    SEMILLAS_FINALES,
    2,
):

    grupos_1 = (
        grupos_validacion_por_semilla[
            semilla_1
        ]
    )

    grupos_2 = (
        grupos_validacion_por_semilla[
            semilla_2
        ]
    )

    ari = adjusted_rand_score(
        grupos_1,
        grupos_2,
    )

    comparaciones_ari.append(
        {
            "semilla_1": semilla_1,
            "semilla_2": semilla_2,
            "ari_validacion": ari,
        }
    )


df_ari_familia_final = pd.DataFrame(
    comparaciones_ari
)


# ----------------------------------------------------------
# Número de grupos efectivos por semilla
# ----------------------------------------------------------

resumen_grupos_semillas = []

for semilla in SEMILLAS_FINALES:

    grupos = (
        grupos_validacion_por_semilla[
            semilla
        ]
    )

    valores, conteos = np.unique(
        grupos,
        return_counts=True,
    )

    resumen_grupos_semillas.append(
        {
            "seed_modelo": semilla,
            "n_grupos_efectivos": len(
                valores
            ),
            "grupo_menor": int(
                conteos.min()
            ),
            "grupo_mayor": int(
                conteos.max()
            ),
            "grupo_menor_pct": (
                100
                * conteos.min()
                / len(grupos)
            ),
        }
    )


df_grupos_semillas = pd.DataFrame(
    resumen_grupos_semillas
)


# ----------------------------------------------------------
# Resumen de estabilidad
# ----------------------------------------------------------

resumen_ari_final = pd.Series(
    {
        "n_comparaciones": len(
            df_ari_familia_final
        ),
        "ari_medio": (
            df_ari_familia_final[
                "ari_validacion"
            ].mean()
        ),
        "ari_minimo": (
            df_ari_familia_final[
                "ari_validacion"
            ].min()
        ),
        "ari_maximo": (
            df_ari_familia_final[
                "ari_validacion"
            ].max()
        ),
    },
    name="resultado",
)


# ----------------------------------------------------------
# Mostrar resultados
# ----------------------------------------------------------

print(
    "Resultados completos por semilla:"
)

display(
    familia_aa_final[
        [
            "seed_modelo",
            "convergio",
            "iteraciones_reales",
            "loss_final",
            "mse_val",
            "purity_val",
            "ami_val",
        ]
    ].round(8)
)


print(
    "\nARI entre pares de semillas:"
)

display(
    df_ari_familia_final.round(8)
)


print(
    "\nResumen del ARI:"
)

display(
    resumen_ari_final.round(8)
)


print(
    "\nGrupos efectivos por semilla:"
)

display(
    df_grupos_semillas.round(6)
)

Resultados completos por semilla:


,seed_modelo,convergio,iteraciones_reales,loss_final,mse_val,purity_val,ami_val
0,0,True,177,128406.224177,0.041781,0.917671,0.451175
1,1,False,300,120419.453863,0.039159,0.925447,0.404173
2,2,True,177,128406.224177,0.041781,0.917671,0.451175
3,3,True,177,128406.224177,0.041781,0.917671,0.451175
4,4,False,300,120419.453014,0.039159,0.925447,0.404173



ARI entre pares de semillas:


,semilla_1,semilla_2,ari_validacion
0,0,1,0.54382
1,0,2,1.00000
2,0,3,1.00000
3,0,4,0.54382
4,1,2,0.54382
5,1,3,0.54382
6,1,4,1.00000
7,2,3,1.00000
8,2,4,0.54382
9,3,4,0.54382



Resumen del ARI:


n_comparaciones    10.000000
ari_medio           0.726292
ari_minimo          0.543820
ari_maximo          1.000000
Name: resultado, dtype: float64


Grupos efectivos por semilla:


,seed_modelo,n_grupos_efectivos,grupo_menor,grupo_mayor,grupo_menor_pct
0,0,5,7,8325,0.034020
1,1,5,1786,5501,8.680016
2,2,5,7,8325,0.034020
3,3,5,7,8325,0.034020
4,4,5,1786,5501,8.680016


In [5]:
# ==========================================================
# Comparar estabilidad de z-score y min-max
# ==========================================================

# ----------------------------------------------------------
# Recuperar la familia anterior con min-max
# ----------------------------------------------------------

familia_aa_minmax = (
    df_registro_actualizado.loc[
        (
            df_registro_actualizado["modelo"]
            == "archetypal_analysis"
        )
        & (
            df_registro_actualizado["dataset"]
            == "medianas_fase_50"
        )
        & (
            df_registro_actualizado["normalizacion"]
            == "minmax"
        )
        & (
            df_registro_actualizado["inicializacion"]
            == "pca_convex_hull_furthest_sum"
        )
        & (
            df_registro_actualizado["K"]
            == 5
        )
        & (
            df_registro_actualizado[
                "seed_modelo"
            ].isin(
                SEMILLAS_FINALES
            )
        )
    ]
    .drop_duplicates(
        subset="experiment_id",
        keep="last",
    )
    .sort_values("seed_modelo")
    .reset_index(drop=True)
)

if len(familia_aa_minmax) != 5:
    raise ValueError(
        "No se encontraron exactamente cinco "
        "ejecuciones de la familia min-max."
    )


# ----------------------------------------------------------
# Función para resumir la estabilidad de una familia
# ----------------------------------------------------------

def resumir_estabilidad_familia(
    familia,
    nombre_familia,
):

    grupos_por_semilla = {}

    for _, fila in familia.iterrows():

        semilla = int(
            fila["seed_modelo"]
        )

        ruta_experimento = (
            RUTA_PROYECTO
            / Path(
                fila["ruta_experimento"]
            )
        )

        grupos_por_semilla[
            semilla
        ] = np.load(
            ruta_experimento
            / "grupos_val.npy"
        )

    comparaciones = []

    for semilla_1, semilla_2 in combinations(
        SEMILLAS_FINALES,
        2,
    ):

        comparaciones.append(
            adjusted_rand_score(
                grupos_por_semilla[
                    semilla_1
                ],
                grupos_por_semilla[
                    semilla_2
                ],
            )
        )

    grupos_menores = []
    porcentajes_menores = []
    grupos_efectivos = []

    for semilla in SEMILLAS_FINALES:

        grupos = (
            grupos_por_semilla[
                semilla
            ]
        )

        _, conteos = np.unique(
            grupos,
            return_counts=True,
        )

        grupos_efectivos.append(
            len(conteos)
        )

        grupos_menores.append(
            int(
                conteos.min()
            )
        )

        porcentajes_menores.append(
            100
            * conteos.min()
            / len(grupos)
        )

    return {
        "familia": nombre_familia,
        "n_ejecuciones": len(
            familia
        ),
        "n_convergieron": int(
            familia["convergio"].sum()
        ),
        "ari_medio": float(
            np.mean(
                comparaciones
            )
        ),
        "ari_minimo": float(
            np.min(
                comparaciones
            )
        ),
        "ari_maximo": float(
            np.max(
                comparaciones
            )
        ),
        "grupos_efectivos_minimo": int(
            np.min(
                grupos_efectivos
            )
        ),
        "grupo_menor_minimo": int(
            np.min(
                grupos_menores
            )
        ),
        "grupo_menor_mediano": float(
            np.median(
                grupos_menores
            )
        ),
        "grupo_menor_pct_minimo": float(
            np.min(
                porcentajes_menores
            )
        ),
        "grupo_menor_pct_mediano": float(
            np.median(
                porcentajes_menores
            )
        ),
    }


# ----------------------------------------------------------
# Comparar ambas familias
# ----------------------------------------------------------

resumen_estabilidad_familias = pd.DataFrame(
    [
        resumir_estabilidad_familia(
            familia=familia_aa_final,
            nombre_familia="zscore",
        ),
        resumir_estabilidad_familia(
            familia=familia_aa_minmax,
            nombre_familia="minmax",
        ),
    ]
)


print(
    "Comparación de estabilidad:"
)

display(
    resumen_estabilidad_familias.round(6)
)


print(
    "\nResultados de la familia min-max:"
)

display(
    familia_aa_minmax[
        [
            "experiment_id",
            "seed_modelo",
            "convergio",
            "iteraciones_reales",
            "mse_val",
            "purity_val",
            "ami_val",
        ]
    ].round(8)
)

Comparación de estabilidad:


,familia,n_ejecuciones,n_convergieron,ari_medio,ari_minimo,ari_maximo,grupos_efectivos_minimo,grupo_menor_minimo,grupo_menor_mediano,grupo_menor_pct_minimo,grupo_menor_pct_mediano
0,zscore,5,3,0.726292,0.543820,1.0,5,7,7.0,0.034020,0.034020
1,minmax,5,3,0.999795,0.999658,1.0,5,508,508.0,2.468896,2.468896



Resultados de la familia min-max:


,experiment_id,seed_modelo,convergio,iteraciones_reales,mse_val,purity_val,ami_val
0,130b477559867762,0,True,234,0.003977,0.928606,0.394516
1,4dd949ada25b888e,1,True,297,0.003977,0.928606,0.394516
2,37dedb2834a007b2,2,False,300,0.003977,0.928606,0.394506
3,349e82cdd54799e0,3,False,300,0.003977,0.928606,0.394506
4,976409505a67e4f6,4,True,204,0.003977,0.928606,0.394516


In [24]:
# ==========================================================
# Registrar la selección definitiva desde archivos guardados
# ==========================================================

from itertools import combinations
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import adjusted_rand_score


# ----------------------------------------------------------
# Rutas
# ----------------------------------------------------------

RUTA_PROYECTO = (
    Path.cwd()
    .resolve()
    .parent
)

RUTA_RESULTADOS_GRILLA = (
    RUTA_PROYECTO
    / "resultados"
    / "busqueda_grilla_aa_5000"
)

RUTA_REGISTRO_GRILLA = (
    RUTA_RESULTADOS_GRILLA
    / "registro_grilla_aa.csv"
)

RUTA_TABLAS_GRILLA = (
    RUTA_RESULTADOS_GRILLA
    / "tablas"
)

RUTA_RESUMEN_BUSQUEDA = (
    RUTA_RESULTADOS_GRILLA
    / "configuracion_busqueda_grilla.json"
)

RUTA_REGISTRO_GENERAL = (
    RUTA_PROYECTO
    / "resultados"
    / "registro_experimentos.csv"
)

RUTA_TABLAS_GRILLA.mkdir(
    parents=True,
    exist_ok=True,
)


# ----------------------------------------------------------
# Función para interpretar columnas booleanas
# ----------------------------------------------------------

def convertir_a_booleano(serie):

    if pd.api.types.is_bool_dtype(
        serie
    ):
        return serie.fillna(False)

    return (
        serie
        .astype(str)
        .str.lower()
        .eq("true")
    )


# ----------------------------------------------------------
# Recuperar las configuraciones válidas de la grilla
# ----------------------------------------------------------

df_grilla = pd.read_csv(
    RUTA_REGISTRO_GRILLA
)

df_grilla = (
    df_grilla
    .drop_duplicates(
        subset="configuracion_id",
        keep="last",
    )
    .reset_index(drop=True)
)

mascara_convergencia_grilla = (
    convertir_a_booleano(
        df_grilla["convergio"]
    )
)

configuraciones_validas = (
    df_grilla.loc[
        df_grilla["estado"].eq("OK")
        & mascara_convergencia_grilla
        & (
            df_grilla[
                "n_grupos_efectivos_val"
            ]
            == df_grilla["K"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Recuperar los dos candidatos con K = 5
# ----------------------------------------------------------

candidato_inicial_zscore = (
    configuraciones_validas.loc[
        (
            configuraciones_validas["representacion"]
            == "medianas_fase_50"
        )
        & (
            configuraciones_validas["normalizacion"]
            == "zscore"
        )
        & (
            configuraciones_validas["inicializacion"]
            == "pca_convex_hull_furthest_sum"
        )
        & (
            configuraciones_validas["K"]
            == 5
        )
    ]
    .copy()
    .reset_index(drop=True)
)

configuracion_final_estable = (
    configuraciones_validas.loc[
        (
            configuraciones_validas["representacion"]
            == "medianas_fase_50"
        )
        & (
            configuraciones_validas["normalizacion"]
            == "minmax"
        )
        & (
            configuraciones_validas["inicializacion"]
            == "pca_convex_hull_furthest_sum"
        )
        & (
            configuraciones_validas["K"]
            == 5
        )
    ]
    .copy()
    .reset_index(drop=True)
)

if len(candidato_inicial_zscore) != 1:
    raise ValueError(
        "No se encontró exactamente un candidato z-score."
    )

if len(configuracion_final_estable) != 1:
    raise ValueError(
        "No se encontró exactamente una configuración "
        "final min-max."
    )


# ----------------------------------------------------------
# Recuperar las familias completas desde el registro general
# ----------------------------------------------------------

df_registro_general = pd.read_csv(
    RUTA_REGISTRO_GENERAL
)

mascara_base = (
    df_registro_general["modelo"].eq(
        "archetypal_analysis"
    )
    & df_registro_general["dataset"].eq(
        "medianas_fase_50"
    )
    & df_registro_general["inicializacion"].eq(
        "pca_convex_hull_furthest_sum"
    )
    & df_registro_general["K"].eq(5)
    & df_registro_general[
        "seed_modelo"
    ].isin(
        [0, 1, 2, 3, 4]
    )
)

familia_zscore = (
    df_registro_general.loc[
        mascara_base
        & df_registro_general[
            "normalizacion"
        ].eq("zscore")
    ]
    .drop_duplicates(
        subset="experiment_id",
        keep="last",
    )
    .sort_values("seed_modelo")
    .reset_index(drop=True)
)

familia_minmax = (
    df_registro_general.loc[
        mascara_base
        & df_registro_general[
            "normalizacion"
        ].eq("minmax")
    ]
    .drop_duplicates(
        subset="experiment_id",
        keep="last",
    )
    .sort_values("seed_modelo")
    .reset_index(drop=True)
)

if len(familia_zscore) != 5:
    raise ValueError(
        "No se encontraron cinco ejecuciones z-score."
    )

if len(familia_minmax) != 5:
    raise ValueError(
        "No se encontraron cinco ejecuciones min-max."
    )


# ----------------------------------------------------------
# Resumir estabilidad a partir de los artefactos
# ----------------------------------------------------------

def resumir_estabilidad_familia(
    familia,
    nombre_familia,
):

    grupos_por_semilla = {}

    for _, fila in familia.iterrows():

        semilla = int(
            fila["seed_modelo"]
        )

        ruta_experimento = (
            RUTA_PROYECTO
            / Path(
                fila["ruta_experimento"]
            )
        )

        ruta_grupos = (
            ruta_experimento
            / "grupos_val.npy"
        )

        if not ruta_grupos.exists():
            raise FileNotFoundError(
                "No se encontraron los grupos en:\n"
                f"{ruta_grupos}"
            )

        grupos_por_semilla[
            semilla
        ] = np.load(
            ruta_grupos
        )

    semillas = sorted(
        grupos_por_semilla
    )

    valores_ari = []

    for semilla_1, semilla_2 in combinations(
        semillas,
        2,
    ):

        valores_ari.append(
            adjusted_rand_score(
                grupos_por_semilla[
                    semilla_1
                ],
                grupos_por_semilla[
                    semilla_2
                ],
            )
        )

    grupos_menores = []
    porcentajes_menores = []
    grupos_efectivos = []

    for semilla in semillas:

        grupos = (
            grupos_por_semilla[
                semilla
            ]
        )

        _, conteos = np.unique(
            grupos,
            return_counts=True,
        )

        grupos_efectivos.append(
            len(conteos)
        )

        grupos_menores.append(
            int(
                conteos.min()
            )
        )

        porcentajes_menores.append(
            100
            * conteos.min()
            / len(grupos)
        )

    convergio = convertir_a_booleano(
        familia["convergio"]
    )

    return {
        "familia": nombre_familia,
        "n_ejecuciones": int(
            len(familia)
        ),
        "n_convergieron": int(
            convergio.sum()
        ),
        "ari_medio": float(
            np.mean(
                valores_ari
            )
        ),
        "ari_minimo": float(
            np.min(
                valores_ari
            )
        ),
        "ari_maximo": float(
            np.max(
                valores_ari
            )
        ),
        "grupos_efectivos_minimo": int(
            np.min(
                grupos_efectivos
            )
        ),
        "grupo_menor_minimo": int(
            np.min(
                grupos_menores
            )
        ),
        "grupo_menor_mediano": float(
            np.median(
                grupos_menores
            )
        ),
        "grupo_menor_pct_minimo": float(
            np.min(
                porcentajes_menores
            )
        ),
        "grupo_menor_pct_mediano": float(
            np.median(
                porcentajes_menores
            )
        ),
    }


resumen_estabilidad_familias = pd.DataFrame(
    [
        resumir_estabilidad_familia(
            familia=familia_zscore,
            nombre_familia="zscore",
        ),
        resumir_estabilidad_familia(
            familia=familia_minmax,
            nombre_familia="minmax",
        ),
    ]
)


# ----------------------------------------------------------
# Guardar las tablas de trazabilidad
# ----------------------------------------------------------

candidato_inicial_zscore.to_csv(
    RUTA_TABLAS_GRILLA
    / "candidato_inicial_zscore.csv",
    index=False,
)

configuracion_final_estable.to_csv(
    RUTA_TABLAS_GRILLA
    / "seleccion_configuracion_aa.csv",
    index=False,
)

resumen_estabilidad_familias.to_csv(
    RUTA_TABLAS_GRILLA
    / "comparacion_estabilidad_finalistas.csv",
    index=False,
)


# ----------------------------------------------------------
# Leer y actualizar el JSON
# ----------------------------------------------------------

if RUTA_RESUMEN_BUSQUEDA.exists():

    with RUTA_RESUMEN_BUSQUEDA.open(
        "r",
        encoding="utf-8",
    ) as archivo_json:

        resumen_busqueda = json.load(
            archivo_json
        )

else:
    resumen_busqueda = {}


fila_zscore = (
    candidato_inicial_zscore
    .iloc[0]
)

fila_final = (
    configuracion_final_estable
    .iloc[0]
)


resumen_busqueda[
    "candidato_inicial_menor_mse"
] = {
    "representacion": str(
        fila_zscore["representacion"]
    ),
    "normalizacion": str(
        fila_zscore["normalizacion"]
    ),
    "inicializacion": str(
        fila_zscore["inicializacion"]
    ),
    "K": int(
        fila_zscore["K"]
    ),
    "mse_val_original": float(
        fila_zscore["mse_val_original"]
    ),
    "ami_val": float(
        fila_zscore["ami_val"]
    ),
    "resultado_estabilidad": (
        "Descartada por sensibilidad a la semilla "
        "y por la aparición de un grupo dominante "
        "de solo siete curvas."
    ),
}


resumen_busqueda[
    "configuracion_seleccionada"
] = {
    "representacion": str(
        fila_final["representacion"]
    ),
    "normalizacion": str(
        fila_final["normalizacion"]
    ),
    "inicializacion": str(
        fila_final["inicializacion"]
    ),
    "K": int(
        fila_final["K"]
    ),
    "mse_val_original": float(
        fila_final["mse_val_original"]
    ),
    "ami_val": float(
        fila_final["ami_val"]
    ),
    "tiempo_entrenamiento_segundos": float(
        fila_final[
            "tiempo_entrenamiento_segundos"
        ]
    ),
    "iteraciones_reales": int(
        fila_final["iteraciones_reales"]
    ),
}


resumen_busqueda[
    "criterio_seleccion"
] = [
    (
        "Medianas de fase con 50 valores presentó "
        "el mejor comportamiento general entre "
        "las representaciones evaluadas."
    ),
    (
        "K=5 fue el último valor con una reducción "
        "sustancial del error de reconstrucción."
    ),
    (
        "Z-score presentó el menor MSE original, "
        "pero fue descartada después de evaluar "
        "cinco semillas sobre el conjunto completo."
    ),
    (
        "Min-max conservó los cinco grupos sin "
        "colapso y alcanzó una estabilidad "
        "prácticamente perfecta."
    ),
    (
        "La configuración min-max ya había sido "
        "evaluada con cinco semillas sobre el "
        "conjunto completo."
    ),
]

resumen_busqueda[
    "comparacion_estabilidad"
] = json.loads(
    resumen_estabilidad_familias.to_json(
        orient="records"
    )
)


with RUTA_RESUMEN_BUSQUEDA.open(
    "w",
    encoding="utf-8",
) as archivo_json:

    json.dump(
        resumen_busqueda,
        archivo_json,
        ensure_ascii=False,
        indent=4,
    )


# ----------------------------------------------------------
# Comprobación final
# ----------------------------------------------------------

print(
    "Configuración definitiva registrada:"
)

display(
    configuracion_final_estable[
        [
            "representacion",
            "normalizacion",
            "inicializacion",
            "K",
            "mse_val_original",
            "ami_val",
            "tiempo_entrenamiento_segundos",
            "iteraciones_reales",
        ]
    ].round(9)
)

print(
    "\nComparación de estabilidad:"
)

display(
    resumen_estabilidad_familias.round(6)
)

print(
    "\nArchivo actualizado:",
    RUTA_RESUMEN_BUSQUEDA,
)

Configuración definitiva registrada:


,representacion,normalizacion,inicializacion,K,mse_val_original,ami_val,tiempo_entrenamiento_segundos,iteraciones_reales
0,medianas_fase_50,minmax,pca_convex_hull_furthest_sum,5,0.000907,0.427747,12.597029,46



Comparación de estabilidad:


,familia,n_ejecuciones,n_convergieron,ari_medio,ari_minimo,ari_maximo,grupos_efectivos_minimo,grupo_menor_minimo,grupo_menor_mediano,grupo_menor_pct_minimo,grupo_menor_pct_mediano
0,zscore,5,3,0.726292,0.543820,1.0,5,7,7.0,0.034020,0.034020
1,minmax,5,3,0.999795,0.999658,1.0,5,508,508.0,2.468896,2.468896



Archivo actualizado: C:\Users\adanm\Code\tesis\resultados\busqueda_grilla_aa_5000\configuracion_busqueda_grilla.json


In [6]:
# ==========================================================
# Ejecutar las dos familias finalistas pendientes
# ==========================================================

from pathlib import Path
import sys

import pandas as pd


# ----------------------------------------------------------
# Rutas e importación del ejecutor
# ----------------------------------------------------------

RUTA_PROYECTO = (
    Path.cwd()
    .resolve()
    .parent
)

RUTA_SRC = (
    RUTA_PROYECTO
    / "src"
)

if not RUTA_SRC.is_dir():
    raise FileNotFoundError(
        "No se encontró la carpeta src en:\n"
        f"{RUTA_SRC}"
    )

if str(RUTA_SRC) not in sys.path:
    sys.path.insert(
        0,
        str(RUTA_SRC),
    )

from experimento_aa import run_aa_experiment


# ----------------------------------------------------------
# Familias pendientes
# ----------------------------------------------------------

DATASET_FINALISTA = "medianas_fase_50"
INICIALIZACION_PENDIENTE = "uniform"
K_FINALISTA = 5

NORMALIZACIONES_PENDIENTES = (
    "zscore",
    "minmax",
)

SEMILLAS_FINALISTAS = (
    0,
    1,
    2,
    3,
    4,
)


# ----------------------------------------------------------
# Ejecutar las diez configuraciones pendientes
# ----------------------------------------------------------

resultados_uniform = []

for normalizacion in NORMALIZACIONES_PENDIENTES:

    for semilla in SEMILLAS_FINALISTAS:

        print(
            "\n"
            + "=" * 70
        )

        print(
            "Normalización:",
            normalizacion,
        )

        print(
            "Inicialización:",
            INICIALIZACION_PENDIENTE,
        )

        print(
            "Semilla:",
            semilla,
        )

        print(
            "=" * 70
        )

        resultado = run_aa_experiment(
            dataset=DATASET_FINALISTA,
            normalizacion=normalizacion,
            k=K_FINALISTA,
            seed=semilla,
            inicializacion=(
                INICIALIZACION_PENDIENTE
            ),
        )

        resultados_uniform.append(
            resultado
        )


# ----------------------------------------------------------
# Recuperar los resultados desde el registro general
# ----------------------------------------------------------

RUTA_REGISTRO_GENERAL = (
    RUTA_PROYECTO
    / "resultados"
    / "registro_experimentos.csv"
)

df_registro_actualizado = pd.read_csv(
    RUTA_REGISTRO_GENERAL
)

familias_uniform = (
    df_registro_actualizado.loc[
        (
            df_registro_actualizado["modelo"]
            == "archetypal_analysis"
        )
        & (
            df_registro_actualizado["dataset"]
            == DATASET_FINALISTA
        )
        & (
            df_registro_actualizado["normalizacion"]
            .isin(
                NORMALIZACIONES_PENDIENTES
            )
        )
        & (
            df_registro_actualizado["inicializacion"]
            == INICIALIZACION_PENDIENTE
        )
        & (
            df_registro_actualizado["K"]
            == K_FINALISTA
        )
        & (
            df_registro_actualizado[
                "seed_modelo"
            ].isin(
                SEMILLAS_FINALISTAS
            )
        )
    ]
    .drop_duplicates(
        subset="experiment_id",
        keep="last",
    )
    .sort_values(
        [
            "normalizacion",
            "seed_modelo",
        ]
    )
    .reset_index(drop=True)
)


if len(familias_uniform) != 10:
    raise ValueError(
        "No se encontraron exactamente diez "
        "ejecuciones de las familias uniform."
    )


# ----------------------------------------------------------
# Mostrar los resultados
# ----------------------------------------------------------

print(
    "\nEjecuciones recuperadas:",
    len(familias_uniform),
)

display(
    familias_uniform[
        [
            "experiment_id",
            "normalizacion",
            "inicializacion",
            "seed_modelo",
            "convergio",
            "iteraciones_reales",
            "tiempo_entrenamiento_segundos",
            "loss_final",
            "mse_val",
            "purity_val",
            "ami_val",
        ]
    ].round(8)
)


Normalización: zscore
Inicialización: uniform
Semilla: 0
Experimento completado: 34ab7e0721eebb83
Tiempo de entrenamiento: 1509.784 segundos
Tiempo total: 1512.155 segundos
RSS validación: 40287.069483

Normalización: zscore
Inicialización: uniform
Semilla: 1
Experimento completado: 3e60ca4e3d51b6e9
Tiempo de entrenamiento: 533.229 segundos
Tiempo total: 535.072 segundos
RSS validación: 40287.136473

Normalización: zscore
Inicialización: uniform
Semilla: 2
Experimento completado: a6c422a0255ddea7
Tiempo de entrenamiento: 1559.404 segundos
Tiempo total: 1561.835 segundos
RSS validación: 40287.069569

Normalización: zscore
Inicialización: uniform
Semilla: 3
Experimento completado: a4de5d2c3ff9652a
Tiempo de entrenamiento: 1529.412 segundos
Tiempo total: 1532.012 segundos
RSS validación: 40287.069728

Normalización: zscore
Inicialización: uniform
Semilla: 4
Experimento completado: 2e60c2347669ff0b
Tiempo de entrenamiento: 1523.506 segundos
Tiempo total: 1525.657 segundos
RSS validación: 

,experiment_id,normalizacion,inicializacion,seed_modelo,convergio,iteraciones_reales,tiempo_entrenamiento_segundos,loss_final,mse_val,purity_val,ami_val
0,cc7d64e07ed4c581,minmax,uniform,0,False,300,785.582564,12245.214849,0.003977,0.928558,0.394342
1,ca2e8b95c574187c,minmax,uniform,1,True,252,744.301793,12042.746947,0.003917,0.929724,0.414947
2,cde5b24cf3f53ed9,minmax,uniform,2,True,212,649.811280,12042.750036,0.003917,0.929724,0.414947
3,36b2d01bfb50934c,minmax,uniform,3,False,300,756.812008,12245.522223,0.003977,0.928606,0.394506
4,ded35b32a1553ef5,minmax,uniform,4,True,215,643.840415,12042.750031,0.003917,0.929724,0.414947
5,34ab7e0721eebb83,zscore,uniform,0,False,300,1509.783679,120419.464851,0.039159,0.925447,0.404173
6,3e60ca4e3d51b6e9,zscore,uniform,1,True,111,533.229331,120419.633247,0.039159,0.925399,0.404095
7,a6c422a0255ddea7,zscore,uniform,2,False,300,1559.404286,120419.465057,0.039159,0.925447,0.404173
8,a4de5d2c3ff9652a,zscore,uniform,3,False,300,1529.412350,120419.465433,0.039159,0.925447,0.404173
9,2e60c2347669ff0b,zscore,uniform,4,False,300,1523.506261,120419.476112,0.039159,0.925447,0.404173


In [7]:
# ==========================================================
# Comparación definitiva de las cuatro familias finalistas
# ==========================================================

from itertools import combinations
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.metrics import adjusted_rand_score


# ----------------------------------------------------------
# Rutas e importaciones
# ----------------------------------------------------------

RUTA_PROYECTO = (
    Path.cwd()
    .resolve()
    .parent
)

if str(RUTA_PROYECTO) not in sys.path:
    sys.path.insert(
        0,
        str(RUTA_PROYECTO),
    )

from src.loaders import cargar_dataset


RUTA_REGISTRO_GENERAL = (
    RUTA_PROYECTO
    / "resultados"
    / "registro_experimentos.csv"
)

df_registro = pd.read_csv(
    RUTA_REGISTRO_GENERAL
)


# ----------------------------------------------------------
# Recuperar validación en su escala original
# ----------------------------------------------------------

datos = cargar_dataset(
    "medianas_fase_50"
)

X_val_original = np.asarray(
    datos["val"]["X"],
    dtype=np.float64,
)


# ----------------------------------------------------------
# Funciones auxiliares
# ----------------------------------------------------------

def convertir_a_booleano(serie):

    if pd.api.types.is_bool_dtype(
        serie
    ):
        return serie.fillna(False)

    return (
        serie
        .astype(str)
        .str.lower()
        .eq("true")
    )


def desnormalizar_reconstruccion(
    X_reconstruida,
    X_original,
    normalizacion,
):

    if normalizacion == "minmax":

        minimo = X_original.min(
            axis=1,
            keepdims=True,
        )

        maximo = X_original.max(
            axis=1,
            keepdims=True,
        )

        escala = maximo - minimo

        return (
            X_reconstruida
            * escala
            + minimo
        )

    if normalizacion == "zscore":

        media = X_original.mean(
            axis=1,
            keepdims=True,
        )

        desviacion = X_original.std(
            axis=1,
            keepdims=True,
            ddof=0,
        )

        return (
            X_reconstruida
            * desviacion
            + media
        )

    raise ValueError(
        "Normalización no soportada: "
        f"{normalizacion}"
    )


def recuperar_familia(
    normalizacion,
    inicializacion,
):

    familia = (
        df_registro.loc[
            (
                df_registro["modelo"]
                == "archetypal_analysis"
            )
            & (
                df_registro["dataset"]
                == "medianas_fase_50"
            )
            & (
                df_registro["normalizacion"]
                == normalizacion
            )
            & (
                df_registro["inicializacion"]
                == inicializacion
            )
            & (
                df_registro["K"]
                == 5
            )
            & (
                df_registro[
                    "seed_modelo"
                ].isin(
                    [0, 1, 2, 3, 4]
                )
            )
        ]
        .drop_duplicates(
            subset="experiment_id",
            keep="last",
        )
        .sort_values("seed_modelo")
        .reset_index(drop=True)
    )

    if len(familia) != 5:
        raise ValueError(
            "No se encontraron cinco ejecuciones para "
            f"{normalizacion} + {inicializacion}."
        )

    return familia


def analizar_familia(
    familia,
    nombre_familia,
):

    grupos_por_semilla = {}
    detalles = []

    for _, fila in familia.iterrows():

        semilla = int(
            fila["seed_modelo"]
        )

        normalizacion = str(
            fila["normalizacion"]
        )

        ruta_experimento = (
            RUTA_PROYECTO
            / Path(
                fila["ruta_experimento"]
            )
        )

        grupos = np.load(
            ruta_experimento
            / "grupos_val.npy"
        )

        alphas = np.load(
            ruta_experimento
            / "alphas_val.npy"
        )

        patrones = np.load(
            ruta_experimento
            / "patrones.npy"
        )

        X_reconstruida_normalizada = (
            alphas
            @ patrones
        )

        X_reconstruida_original = (
            desnormalizar_reconstruccion(
                X_reconstruida=(
                    X_reconstruida_normalizada
                ),
                X_original=X_val_original,
                normalizacion=normalizacion,
            )
        )

        mse_original = float(
            np.mean(
                (
                    X_val_original
                    - X_reconstruida_original
                )
                ** 2
            )
        )

        _, conteos = np.unique(
            grupos,
            return_counts=True,
        )

        grupos_por_semilla[
            semilla
        ] = grupos

        detalles.append(
            {
                "familia": nombre_familia,
                "normalizacion": normalizacion,
                "inicializacion": str(
                    fila["inicializacion"]
                ),
                "seed_modelo": semilla,
                "convergio": bool(
                    convertir_a_booleano(
                        pd.Series(
                            [
                                fila["convergio"]
                            ]
                        )
                    ).iloc[0]
                ),
                "iteraciones_reales": int(
                    fila[
                        "iteraciones_reales"
                    ]
                ),
                "tiempo_segundos": float(
                    fila[
                        "tiempo_entrenamiento_segundos"
                    ]
                ),
                "mse_original": mse_original,
                "purity_val": float(
                    fila["purity_val"]
                ),
                "ami_val": float(
                    fila["ami_val"]
                ),
                "n_grupos_efectivos": int(
                    len(conteos)
                ),
                "grupo_menor": int(
                    conteos.min()
                ),
                "grupo_menor_pct": float(
                    100
                    * conteos.min()
                    / len(grupos)
                ),
            }
        )

    valores_ari = []

    semillas = sorted(
        grupos_por_semilla
    )

    for semilla_1, semilla_2 in combinations(
        semillas,
        2,
    ):

        valores_ari.append(
            adjusted_rand_score(
                grupos_por_semilla[
                    semilla_1
                ],
                grupos_por_semilla[
                    semilla_2
                ],
            )
        )

    detalle = pd.DataFrame(
        detalles
    )

    resumen = {
        "familia": nombre_familia,
        "n_convergieron": int(
            detalle["convergio"].sum()
        ),
        "ari_medio": float(
            np.mean(
                valores_ari
            )
        ),
        "ari_minimo": float(
            np.min(
                valores_ari
            )
        ),
        "mse_original_medio": float(
            detalle[
                "mse_original"
            ].mean()
        ),
        "mse_original_std": float(
            detalle[
                "mse_original"
            ].std()
        ),
        "ami_medio": float(
            detalle[
                "ami_val"
            ].mean()
        ),
        "ami_std": float(
            detalle[
                "ami_val"
            ].std()
        ),
        "tiempo_mediano_segundos": float(
            detalle[
                "tiempo_segundos"
            ].median()
        ),
        "grupo_menor_minimo": int(
            detalle[
                "grupo_menor"
            ].min()
        ),
        "grupo_menor_pct_minimo": float(
            detalle[
                "grupo_menor_pct"
            ].min()
        ),
    }

    return (
        detalle,
        resumen,
    )


# ----------------------------------------------------------
# Recuperar las cuatro familias
# ----------------------------------------------------------

familias = {
    "minmax_geometrica": recuperar_familia(
        normalizacion="minmax",
        inicializacion=(
            "pca_convex_hull_furthest_sum"
        ),
    ),
    "minmax_uniform": recuperar_familia(
        normalizacion="minmax",
        inicializacion="uniform",
    ),
    "zscore_geometrica": recuperar_familia(
        normalizacion="zscore",
        inicializacion=(
            "pca_convex_hull_furthest_sum"
        ),
    ),
    "zscore_uniform": recuperar_familia(
        normalizacion="zscore",
        inicializacion="uniform",
    ),
}


# ----------------------------------------------------------
# Analizar las cuatro familias
# ----------------------------------------------------------

detalles_familias = []
resumenes_familias = []

for nombre_familia, familia in familias.items():

    detalle, resumen = analizar_familia(
        familia=familia,
        nombre_familia=nombre_familia,
    )

    detalles_familias.append(
        detalle
    )

    resumenes_familias.append(
        resumen
    )


df_detalle_finalistas = pd.concat(
    detalles_familias,
    ignore_index=True,
)

df_resumen_finalistas = pd.DataFrame(
    resumenes_familias
)


# ----------------------------------------------------------
# Mostrar resultados
# ----------------------------------------------------------

print(
    "Resumen de las cuatro familias:"
)

display(
    df_resumen_finalistas[
        [
            "familia",
            "n_convergieron",
            "ari_medio",
            "ari_minimo",
            "mse_original_medio",
            "mse_original_std",
            "ami_medio",
            "ami_std",
            "tiempo_mediano_segundos",
            "grupo_menor_minimo",
            "grupo_menor_pct_minimo",
        ]
    ]
    .sort_values(
        [
            "ari_medio",
            "mse_original_medio",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .round(8)
)


print(
    "\nDetalle por semilla:"
)

display(
    df_detalle_finalistas[
        [
            "familia",
            "seed_modelo",
            "convergio",
            "iteraciones_reales",
            "mse_original",
            "ami_val",
            "grupo_menor",
            "grupo_menor_pct",
        ]
    ]
    .sort_values(
        [
            "familia",
            "seed_modelo",
        ]
    )
    .round(8)
)

Resumen de las cuatro familias:


,familia,n_convergieron,ari_medio,ari_minimo,mse_original_medio,mse_original_std,ami_medio,ami_std,tiempo_mediano_segundos,grupo_menor_minimo,grupo_menor_pct_minimo
3,zscore_uniform,1,0.999822,0.999554,0.000579,1.000000e-08,0.404157,0.000035,1523.506261,1786,8.680016
0,minmax_geometrica,3,0.999795,0.999658,0.000796,1.000000e-08,0.394512,0.000005,735.416407,508,2.468896
2,zscore_geometrica,3,0.726292,0.543820,0.000611,2.970000e-05,0.432374,0.025744,656.260943,7,0.034020
1,minmax_uniform,3,0.676557,0.460973,0.000777,1.783000e-05,0.406738,0.011241,744.301793,509,2.473756



Detalle por semilla:


,familia,seed_modelo,convergio,iteraciones_reales,mse_original,ami_val,grupo_menor,grupo_menor_pct
0,minmax_geometrica,0,True,234,0.000796,0.394516,508,2.468896
1,minmax_geometrica,1,True,297,0.000796,0.394516,508,2.468896
2,minmax_geometrica,2,False,300,0.000796,0.394506,509,2.473756
3,minmax_geometrica,3,False,300,0.000796,0.394506,509,2.473756
4,minmax_geometrica,4,True,204,0.000796,0.394516,508,2.468896
5,minmax_uniform,0,False,300,0.000796,0.394342,510,2.478616
6,minmax_uniform,1,True,252,0.000764,0.414947,1080,5.248834
7,minmax_uniform,2,True,212,0.000764,0.414947,1080,5.248834
8,minmax_uniform,3,False,300,0.000796,0.394506,509,2.473756
9,minmax_uniform,4,True,215,0.000764,0.414947,1080,5.248834


In [8]:
# ==========================================================
# Confirmación final con un máximo de 600 iteraciones
# ==========================================================

from pathlib import Path
import sys

import pandas as pd


# ----------------------------------------------------------
# Rutas e importación del módulo
# ----------------------------------------------------------

RUTA_PROYECTO = (
    Path.cwd()
    .resolve()
    .parent
)

RUTA_SRC = (
    RUTA_PROYECTO
    / "src"
)

if not RUTA_SRC.is_dir():
    raise FileNotFoundError(
        "No se encontró la carpeta src en:\n"
        f"{RUTA_SRC}"
    )

if str(RUTA_SRC) not in sys.path:
    sys.path.insert(
        0,
        str(RUTA_SRC),
    )

import experimento_aa


# ----------------------------------------------------------
# Cambiar explícitamente el límite externo
# ----------------------------------------------------------

MAX_ITER_CONFIRMACION = 600

experimento_aa.AA_MAX_ITER = (
    MAX_ITER_CONFIRMACION
)

print(
    "Máximo de iteraciones configurado:",
    experimento_aa.AA_MAX_ITER,
)


# ----------------------------------------------------------
# Configuración candidata
# ----------------------------------------------------------

DATASET_CONFIRMACION = "medianas_fase_50"
NORMALIZACION_CONFIRMACION = "zscore"
INICIALIZACION_CONFIRMACION = "uniform"
K_CONFIRMACION = 5

SEMILLAS_CONFIRMACION = (
    0,
    1,
    2,
    3,
    4,
)


# ----------------------------------------------------------
# Ejecutar las cinco semillas
# ----------------------------------------------------------

resultados_confirmacion_600 = []

for semilla in SEMILLAS_CONFIRMACION:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "Confirmación con 600 iteraciones"
    )

    print(
        "Semilla:",
        semilla,
    )

    print(
        "=" * 70
    )

    resultado = (
        experimento_aa.run_aa_experiment(
            dataset=DATASET_CONFIRMACION,
            normalizacion=(
                NORMALIZACION_CONFIRMACION
            ),
            k=K_CONFIRMACION,
            seed=semilla,
            inicializacion=(
                INICIALIZACION_CONFIRMACION
            ),
        )
    )

    resultados_confirmacion_600.append(
        resultado
    )


# ----------------------------------------------------------
# Recuperar la nueva familia desde el registro
# ----------------------------------------------------------

RUTA_REGISTRO_GENERAL = (
    RUTA_PROYECTO
    / "resultados"
    / "registro_experimentos.csv"
)

df_registro_actualizado = pd.read_csv(
    RUTA_REGISTRO_GENERAL
)

familia_confirmacion_600 = (
    df_registro_actualizado.loc[
        (
            df_registro_actualizado["modelo"]
            == "archetypal_analysis"
        )
        & (
            df_registro_actualizado["dataset"]
            == DATASET_CONFIRMACION
        )
        & (
            df_registro_actualizado["normalizacion"]
            == NORMALIZACION_CONFIRMACION
        )
        & (
            df_registro_actualizado["inicializacion"]
            == INICIALIZACION_CONFIRMACION
        )
        & (
            df_registro_actualizado["K"]
            == K_CONFIRMACION
        )
        & (
            df_registro_actualizado["max_iter"]
            == MAX_ITER_CONFIRMACION
        )
        & (
            df_registro_actualizado[
                "seed_modelo"
            ].isin(
                SEMILLAS_CONFIRMACION
            )
        )
    ]
    .drop_duplicates(
        subset="experiment_id",
        keep="last",
    )
    .sort_values("seed_modelo")
    .reset_index(drop=True)
)


if len(familia_confirmacion_600) != 5:
    raise ValueError(
        "No se encontraron exactamente cinco "
        "ejecuciones con max_iter=600."
    )


# ----------------------------------------------------------
# Mostrar resultados
# ----------------------------------------------------------

print(
    "\nEjecuciones recuperadas:",
    len(familia_confirmacion_600),
)

display(
    familia_confirmacion_600[
        [
            "experiment_id",
            "seed_modelo",
            "max_iter",
            "convergio",
            "iteraciones_reales",
            "tiempo_entrenamiento_segundos",
            "loss_final",
            "mse_val",
            "purity_val",
            "ami_val",
        ]
    ].round(8)
)

Máximo de iteraciones configurado: 600

Confirmación con 600 iteraciones
Semilla: 0
Experimento completado: 0b2e03bff00931dc
Tiempo de entrenamiento: 1609.602 segundos
Tiempo total: 1612.086 segundos
RSS validación: 40287.067790

Confirmación con 600 iteraciones
Semilla: 1
Experimento completado: c4528e3b65141254
Tiempo de entrenamiento: 564.387 segundos
Tiempo total: 566.026 segundos
RSS validación: 40287.136473

Confirmación con 600 iteraciones
Semilla: 2
Experimento completado: 098a61214877e60e
Tiempo de entrenamiento: 1673.828 segundos
Tiempo total: 1676.108 segundos
RSS validación: 40287.067781

Confirmación con 600 iteraciones
Semilla: 3
Experimento completado: 1e145980f90d8803
Tiempo de entrenamiento: 1623.565 segundos
Tiempo total: 1625.502 segundos
RSS validación: 40287.067754

Confirmación con 600 iteraciones
Semilla: 4
Experimento completado: dafb70f43b2d4850
Tiempo de entrenamiento: 1821.443 segundos
Tiempo total: 1823.486 segundos
RSS validación: 40287.067788

Ejecuciones 

,experiment_id,seed_modelo,max_iter,convergio,iteraciones_reales,tiempo_entrenamiento_segundos,loss_final,mse_val,purity_val,ami_val
0,0b2e03bff00931dc,0,600,True,324,1609.601606,120419.460885,0.039159,0.925447,0.404173
1,c4528e3b65141254,1,600,True,111,564.387402,120419.633247,0.039159,0.925399,0.404095
2,098a61214877e60e,2,600,True,325,1673.827824,120419.460863,0.039159,0.925447,0.404173
3,1e145980f90d8803,3,600,True,327,1623.564696,120419.460801,0.039159,0.925447,0.404173
4,dafb70f43b2d4850,4,600,True,348,1821.442845,120419.460880,0.039159,0.925447,0.404173


In [9]:
# ==========================================================
# Estabilidad final de z-score + uniform con 600 iteraciones
# ==========================================================

from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import adjusted_rand_score


# ----------------------------------------------------------
# Cargar asignaciones y calcular resultados por semilla
# ----------------------------------------------------------

grupos_por_semilla_600 = {}
detalle_confirmacion_600 = []

for _, fila in familia_confirmacion_600.iterrows():

    semilla = int(
        fila["seed_modelo"]
    )

    ruta_experimento = (
        RUTA_PROYECTO
        / Path(
            fila["ruta_experimento"]
        )
    )

    grupos = np.load(
        ruta_experimento
        / "grupos_val.npy"
    )

    _, conteos = np.unique(
        grupos,
        return_counts=True,
    )

    grupos_por_semilla_600[
        semilla
    ] = grupos

    detalle_confirmacion_600.append(
        {
            "experiment_id": (
                fila["experiment_id"]
            ),
            "seed_modelo": semilla,
            "convergio": bool(
                fila["convergio"]
            ),
            "iteraciones_reales": int(
                fila[
                    "iteraciones_reales"
                ]
            ),
            "mse_val": float(
                fila["mse_val"]
            ),
            "ami_val": float(
                fila["ami_val"]
            ),
            "n_grupos_efectivos": int(
                len(conteos)
            ),
            "grupo_menor": int(
                conteos.min()
            ),
            "grupo_menor_pct": float(
                100
                * conteos.min()
                / len(grupos)
            ),
        }
    )


df_detalle_confirmacion_600 = pd.DataFrame(
    detalle_confirmacion_600
)


# ----------------------------------------------------------
# ARI entre todas las parejas de semillas
# ----------------------------------------------------------

comparaciones_ari_600 = []

for semilla_1, semilla_2 in combinations(
    sorted(
        grupos_por_semilla_600
    ),
    2,
):

    ari = adjusted_rand_score(
        grupos_por_semilla_600[
            semilla_1
        ],
        grupos_por_semilla_600[
            semilla_2
        ],
    )

    comparaciones_ari_600.append(
        {
            "semilla_1": semilla_1,
            "semilla_2": semilla_2,
            "ari_validacion": ari,
        }
    )


df_ari_confirmacion_600 = pd.DataFrame(
    comparaciones_ari_600
)


# ----------------------------------------------------------
# Resumen de la familia definitiva
# ----------------------------------------------------------

resumen_confirmacion_600 = pd.Series(
    {
        "n_ejecuciones": len(
            df_detalle_confirmacion_600
        ),
        "n_convergieron": int(
            df_detalle_confirmacion_600[
                "convergio"
            ].sum()
        ),
        "ari_medio": (
            df_ari_confirmacion_600[
                "ari_validacion"
            ].mean()
        ),
        "ari_minimo": (
            df_ari_confirmacion_600[
                "ari_validacion"
            ].min()
        ),
        "ari_maximo": (
            df_ari_confirmacion_600[
                "ari_validacion"
            ].max()
        ),
        "grupo_menor_minimo": int(
            df_detalle_confirmacion_600[
                "grupo_menor"
            ].min()
        ),
        "grupo_menor_pct_minimo": (
            df_detalle_confirmacion_600[
                "grupo_menor_pct"
            ].min()
        ),
    },
    name="resultado",
)


# ----------------------------------------------------------
# Mostrar resultados
# ----------------------------------------------------------

print(
    "Estabilidad de la familia definitiva:"
)

display(
    resumen_confirmacion_600.round(8)
)

print(
    "\nARI entre semillas:"
)

display(
    df_ari_confirmacion_600.round(8)
)

print(
    "\nDetalle por semilla:"
)

display(
    df_detalle_confirmacion_600.round(8)
)

Estabilidad de la familia definitiva:


n_ejecuciones                5.000000
n_convergieron               5.000000
ari_medio                    0.999822
ari_minimo                   0.999554
ari_maximo                   1.000000
grupo_menor_minimo        1786.000000
grupo_menor_pct_minimo       8.680016
Name: resultado, dtype: float64


ARI entre semillas:


,semilla_1,semilla_2,ari_validacion
0,0,1,0.999554
1,0,2,1.000000
2,0,3,1.000000
3,0,4,1.000000
4,1,2,0.999554
5,1,3,0.999554
6,1,4,0.999554
7,2,3,1.000000
8,2,4,1.000000
9,3,4,1.000000



Detalle por semilla:


,experiment_id,seed_modelo,convergio,iteraciones_reales,mse_val,ami_val,n_grupos_efectivos,grupo_menor,grupo_menor_pct
0,0b2e03bff00931dc,0,True,324,0.039159,0.404173,5,1786,8.680016
1,c4528e3b65141254,1,True,111,0.039159,0.404095,5,1787,8.684876
2,098a61214877e60e,2,True,325,0.039159,0.404173,5,1786,8.680016
3,1e145980f90d8803,3,True,327,0.039159,0.404173,5,1786,8.680016
4,dafb70f43b2d4850,4,True,348,0.039159,0.404173,5,1786,8.680016


In [10]:
# ==========================================================
# Guardar la familia AA seleccionada definitivamente
# ==========================================================

from pathlib import Path
import json

import pandas as pd


# ----------------------------------------------------------
# Rutas
# ----------------------------------------------------------

RUTA_RESULTADOS_GRILLA = (
    RUTA_PROYECTO
    / "resultados"
    / "busqueda_grilla_aa_5000"
)

RUTA_TABLAS_GRILLA = (
    RUTA_RESULTADOS_GRILLA
    / "tablas"
)

RUTA_TABLAS_GRILLA.mkdir(
    parents=True,
    exist_ok=True,
)

RUTA_RESUMEN_BUSQUEDA = (
    RUTA_RESULTADOS_GRILLA
    / "configuracion_busqueda_grilla.json"
)


# ----------------------------------------------------------
# Resumen definitivo
# ----------------------------------------------------------

resumen_familia_final = pd.DataFrame(
    [
        {
            "dataset": "medianas_fase_50",
            "normalizacion": "zscore",
            "inicializacion": "uniform",
            "K": 5,
            "optimizador": "nnls",
            "max_iter": 600,
            "tol": 1e-4,
            "n_semillas": 5,
            "n_convergieron": int(
                resumen_confirmacion_600[
                    "n_convergieron"
                ]
            ),
            "ari_medio": float(
                resumen_confirmacion_600[
                    "ari_medio"
                ]
            ),
            "ari_minimo": float(
                resumen_confirmacion_600[
                    "ari_minimo"
                ]
            ),
            "ari_maximo": float(
                resumen_confirmacion_600[
                    "ari_maximo"
                ]
            ),
            "mse_val_media": float(
                familia_confirmacion_600[
                    "mse_val"
                ].mean()
            ),
            "mse_val_std": float(
                familia_confirmacion_600[
                    "mse_val"
                ].std()
            ),
            "ami_val_media": float(
                familia_confirmacion_600[
                    "ami_val"
                ].mean()
            ),
            "ami_val_std": float(
                familia_confirmacion_600[
                    "ami_val"
                ].std()
            ),
            "grupo_menor_minimo": int(
                resumen_confirmacion_600[
                    "grupo_menor_minimo"
                ]
            ),
            "grupo_menor_pct_minimo": float(
                resumen_confirmacion_600[
                    "grupo_menor_pct_minimo"
                ]
            ),
        }
    ]
)


# ----------------------------------------------------------
# Guardar tablas
# ----------------------------------------------------------

familia_confirmacion_600.to_csv(
    RUTA_TABLAS_GRILLA
    / "ejecuciones_familia_aa_final.csv",
    index=False,
)

df_detalle_confirmacion_600.to_csv(
    RUTA_TABLAS_GRILLA
    / "detalle_familia_aa_final.csv",
    index=False,
)

df_ari_confirmacion_600.to_csv(
    RUTA_TABLAS_GRILLA
    / "ari_familia_aa_final.csv",
    index=False,
)

resumen_familia_final.to_csv(
    RUTA_TABLAS_GRILLA
    / "seleccion_configuracion_aa.csv",
    index=False,
)


# ----------------------------------------------------------
# Actualizar el JSON de trazabilidad
# ----------------------------------------------------------

if RUTA_RESUMEN_BUSQUEDA.exists():

    with RUTA_RESUMEN_BUSQUEDA.open(
        "r",
        encoding="utf-8",
    ) as archivo_json:

        resumen_busqueda = json.load(
            archivo_json
        )

else:
    resumen_busqueda = {}


resumen_busqueda[
    "configuracion_seleccionada"
] = {
    "representacion": "medianas_fase_50",
    "normalizacion": "zscore",
    "inicializacion": "uniform",
    "K": 5,
    "optimizador": "nnls",
    "max_iter": 600,
    "tol": 1e-4,
    "semillas": [
        0,
        1,
        2,
        3,
        4,
    ],
    "experiment_ids": (
        familia_confirmacion_600[
            "experiment_id"
        ]
        .astype(str)
        .tolist()
    ),
}


resumen_busqueda[
    "confirmacion_configuracion_final"
] = {
    "n_ejecuciones": 5,
    "n_convergieron": int(
        resumen_confirmacion_600[
            "n_convergieron"
        ]
    ),
    "ari_medio": float(
        resumen_confirmacion_600[
            "ari_medio"
        ]
    ),
    "ari_minimo": float(
        resumen_confirmacion_600[
            "ari_minimo"
        ]
    ),
    "ari_maximo": float(
        resumen_confirmacion_600[
            "ari_maximo"
        ]
    ),
    "grupo_menor_minimo": int(
        resumen_confirmacion_600[
            "grupo_menor_minimo"
        ]
    ),
    "grupo_menor_pct_minimo": float(
        resumen_confirmacion_600[
            "grupo_menor_pct_minimo"
        ]
    ),
}


resumen_busqueda[
    "criterio_seleccion"
] = [
    (
        "La búsqueda conjunta evaluó representación, "
        "normalización, inicialización y K."
    ),
    (
        "K=5 fue el último valor con una reducción "
        "sustancial del error de reconstrucción."
    ),
    (
        "Las cuatro familias finalistas con K=5 "
        "fueron evaluadas con cinco semillas."
    ),
    (
        "Z-score con inicialización uniform obtuvo "
        "el menor MSE original y una estabilidad "
        "prácticamente perfecta."
    ),
    (
        "El máximo externo aumentó de 300 a 600 "
        "porque cuatro semillas necesitaron entre "
        "324 y 348 iteraciones para converger."
    ),
]


with RUTA_RESUMEN_BUSQUEDA.open(
    "w",
    encoding="utf-8",
) as archivo_json:

    json.dump(
        resumen_busqueda,
        archivo_json,
        ensure_ascii=False,
        indent=4,
    )


# ----------------------------------------------------------
# Comprobar
# ----------------------------------------------------------

print(
    "Configuración definitiva guardada:"
)

display(
    resumen_familia_final.round(8)
)

print(
    "\nDirectorio:",
    RUTA_TABLAS_GRILLA,
)

Configuración definitiva guardada:


,dataset,normalizacion,inicializacion,K,optimizador,max_iter,tol,n_semillas,n_convergieron,ari_medio,ari_minimo,ari_maximo,mse_val_media,mse_val_std,ami_val_media,ami_val_std,grupo_menor_minimo,grupo_menor_pct_minimo
0,medianas_fase_50,zscore,uniform,5,nnls,600,0.0001,5,5,0.999822,0.999554,1.0,0.039159,3.000000e-08,0.404157,0.000035,1786,8.680016



Directorio: C:\Users\adanm\Code\tesis\resultados\busqueda_grilla_aa_5000\tablas
